In [1]:
# ============================================================
# S1 REDO — CELL 1
# Purpose:
#   1. Initialize the runtime and read the HF Kaggle Secret.
#   2. Download the official Phase 2 dataset from Hugging Face.
#   3. Locate Architecture A checkpoint/configuration files.
#   4. Inspect dataset schema and report paths for the next cell.
# ============================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import torch

S1_SEED = 42
random.seed(S1_SEED)
np.random.seed(S1_SEED)
torch.manual_seed(S1_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(S1_SEED)

S1_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 68)
print("S1 REDO — ENVIRONMENT")
print("=" * 68)
print("Python/PyTorch:", torch.__version__)
print("Device:", S1_DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")

# ---------- Read the Hugging Face token from Kaggle Secrets ----------
try:
    from kaggle_secrets import UserSecretsClient
    S1_HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception as exc:
    raise RuntimeError(
        "Could not read the Kaggle Secret named 'HF'. "
        "In Kaggle, add your Hugging Face access token under Add-ons > Secrets "
        "with the exact name HF, then rerun this cell."
    ) from exc

if not S1_HF_TOKEN:
    raise RuntimeError("The Kaggle Secret 'HF' is empty.")

print("HF secret: available (token will not be displayed)")

# ---------- Download Phase 2 data ----------
from huggingface_hub import snapshot_download

S1_DATA_LOCAL_DIR = Path("/kaggle/working/S1_phase2_hf")
S1_DATA_DIR = S1_DATA_LOCAL_DIR / "Phase_2_LNER"

S1_REQUIRED_DATA_FILES = [
    S1_DATA_DIR / "Train" / "fold1-text.json",
    S1_DATA_DIR / "Train" / "fold1-gold.json",
    S1_DATA_DIR / "Train" / "fold2-text.json",
    S1_DATA_DIR / "Train" / "fold2-gold.json",
    S1_DATA_DIR / "Train" / "labels.txt",
    S1_DATA_DIR / "Test" / "fold3-text.json",
]

if not all(p.is_file() for p in S1_REQUIRED_DATA_FILES):
    print("\nDownloading official Phase 2 dataset from Hugging Face...")
    snapshot_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        allow_patterns=["Phase_2_LNER/**"],
        local_dir=str(S1_DATA_LOCAL_DIR),
        token=S1_HF_TOKEN,
    )
else:
    print("\nPhase 2 files already exist locally; download skipped.")

S1_MISSING_FILES = [str(p) for p in S1_REQUIRED_DATA_FILES if not p.is_file()]
if S1_MISSING_FILES:
    raise FileNotFoundError(
        "The download did not produce all expected files:\n"
        + "\n".join(S1_MISSING_FILES)
    )

# ---------- Inspect official labels and dataset schema ----------
with open(S1_DATA_DIR / "Train" / "fold1-text.json", "r", encoding="utf-8") as f:
    S1_FOLD1_TEXT = json.load(f)

with open(S1_DATA_DIR / "Train" / "fold1-gold.json", "r", encoding="utf-8") as f:
    S1_FOLD1_GOLD = json.load(f)

with open(S1_DATA_DIR / "Train" / "fold2-text.json", "r", encoding="utf-8") as f:
    S1_FOLD2_TEXT = json.load(f)

with open(S1_DATA_DIR / "Train" / "fold2-gold.json", "r", encoding="utf-8") as f:
    S1_FOLD2_GOLD = json.load(f)

S1_LABELS = [
    line.strip()
    for line in (S1_DATA_DIR / "Train" / "labels.txt").read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

S1_TEST_JSON_PATH = S1_DATA_DIR / "Test" / "fold3-text.json"

with open(S1_TEST_JSON_PATH, "r", encoding="utf-8") as f:
    S1_TEST_TEXTS_CHECK = json.load(f)

print("\n" + "=" * 68)
print("PHASE 2 DATA CHECK")
print("=" * 68)
print("Fold 1 text documents:", len(S1_FOLD1_TEXT))
print("Fold 1 gold documents:", len(S1_FOLD1_GOLD))
print("Fold 2 text documents:", len(S1_FOLD2_TEXT))
print("Fold 2 gold documents:", len(S1_FOLD2_GOLD))
print("Fold 3 test documents:", len(S1_TEST_TEXTS_CHECK))
print("Official labels:", S1_LABELS)
print("Fold 1 text/gold IDs match:", set(S1_FOLD1_TEXT) == set(S1_FOLD1_GOLD))
print("Fold 2 text/gold IDs match:", set(S1_FOLD2_TEXT) == set(S1_FOLD2_GOLD))
print("Sample document ID:", next(iter(S1_FOLD1_TEXT)))
print("Sample document length:", len(next(iter(S1_FOLD1_TEXT.values()))))
print("Fold 3 gold labels accessed: NO")

# ---------- Discover the Phase 1 Architecture A assets ----------
S1_INPUT_ROOT = Path("/kaggle/input")
S1_ARCH_A_DIRS = []

if S1_INPUT_ROOT.exists():
    for p in S1_INPUT_ROOT.rglob("architecture_A"):
        if p.is_dir():
            has_checkpoint = (
                (p / "checkpoints" / "best.pt").is_file()
                or (p / "model_A_final.pt").is_file()
                or (p / "model_A_state_dict.pt").is_file()
            )
            if has_checkpoint:
                S1_ARCH_A_DIRS.append(p)

S1_ARCH_A_DIRS = sorted(set(S1_ARCH_A_DIRS), key=lambda p: str(p))

print("\n" + "=" * 68)
print("ARCHITECTURE A DISCOVERY")
print("=" * 68)

if not S1_ARCH_A_DIRS:
    print("No Architecture A directory with a recognized checkpoint was found.")
    print("Available input dataset folders:")
    if S1_INPUT_ROOT.exists():
        for p in S1_INPUT_ROOT.iterdir():
            print(" ", p)
else:
    for i, arch_dir in enumerate(S1_ARCH_A_DIRS, start=1):
        print(f"\nCandidate {i}: {arch_dir}")
        for rel in [
            "checkpoints/best.pt",
            "checkpoints/latest.pt",
            "configs",
            "model_A_final.pt",
            "model_A_metadata.json",
            "model_A_state_dict.pt",
        ]:
            p = arch_dir / rel
            if p.is_file():
                print(f"  FILE  {rel} ({p.stat().st_size / 1024**2:.2f} MiB)")
            elif p.is_dir():
                print(f"  DIR   {rel}")
                for child in sorted(p.iterdir())[:20]:
                    if child.is_file():
                        print(
                            f"        {child.name} "
                            f"({child.stat().st_size / 1024:.1f} KiB)"
                        )

# ---------- Discover the Phase 1 tokenizer ----------
S1_TOKENIZER_CANDIDATES = []
if S1_INPUT_ROOT.exists():
    for p in S1_INPUT_ROOT.rglob("tokenizer.json"):
        if p.is_file():
            S1_TOKENIZER_CANDIDATES.append(p)

print("\nTokenizer files:")
if S1_TOKENIZER_CANDIDATES:
    for p in sorted(S1_TOKENIZER_CANDIDATES, key=lambda p: str(p))[:25]:
        print(f"  {p} ({p.stat().st_size / 1024**2:.2f} MiB)")
else:
    print("No tokenizer.json found in Kaggle inputs.")

# ---------- Preview small JSON configuration/metadata files ----------
S1_METADATA_FILES = []
for arch_dir in S1_ARCH_A_DIRS:
    for p in arch_dir.rglob("*.json"):
        if p.is_file() and p.stat().st_size <= 100_000:
            S1_METADATA_FILES.append(p)

print("\nArchitecture A configuration/metadata previews:")
if not S1_METADATA_FILES:
    print("No small JSON configuration files found under Architecture A.")
else:
    for p in sorted(S1_METADATA_FILES, key=lambda p: str(p))[:12]:
        print(f"\n--- {p} ---")
        try:
            obj = json.loads(p.read_text(encoding="utf-8"))
            if isinstance(obj, dict):
                print(json.dumps(obj, indent=2, ensure_ascii=False)[:5000])
            else:
                print(json.dumps(obj, indent=2, ensure_ascii=False)[:2000])
        except Exception as exc:
            print("Could not parse this JSON:", type(exc).__name__, str(exc)[:200])

print("\nCell 1 finished. No model has been trained or modified.")

S1 REDO — ENVIRONMENT
Python/PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GiB
HF secret: available (token will not be displayed)



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]


PHASE 2 DATA CHECK
Fold 1 text documents: 35
Fold 1 gold documents: 35
Fold 2 text documents: 35
Fold 2 gold documents: 35
Fold 3 test documents: 35
Official labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Fold 1 text/gold IDs match: True
Fold 2 text/gold IDs match: True
Sample document ID: 115651329
Sample document length: 18616
Fold 3 gold labels accessed: NO

ARCHITECTURE A DISCOVERY

Candidate 1: /kaggle/input/datasets/rohitbatra2004/phase111/architecture_A
  FILE  checkpoints/best.pt (1431.08 MiB)
  FILE  checkpoints/latest.pt (1431.08 MiB)
  DIR   configs
        architecture_A_data_manifest.json (1.9 KiB)
        architecture_A_generation_config.json (0.5 KiB)
        architecture_A_optimization_manifest.json (0.6 KiB)
        official_test_manifest_A.json (11.2 KiB)
  FILE  model_A_final.pt (1431.08 MiB)
  FILE  model_A_metadata.json (0.00 MiB)
  FILE  model_A_state_dict.pt (477.01 MiB)

Tokenizer files:
  /k

In [2]:
# ============================================================
# S1 REDO — CELL 2
# Purpose:
#   Inspect Architecture A metadata and pretrained state dict.
#   Verify parameter count, tensor names, shapes, and tied weights.
# ============================================================

import json
import torch
from pathlib import Path

if not S1_ARCH_A_DIRS:
    raise FileNotFoundError(
        "Architecture A checkpoint was not found in Cell 1."
    )

# Use the Architecture A directory discovered in Cell 1.
S1_ARCH_A_DIR = S1_ARCH_A_DIRS[0]
S1_A_METADATA_PATH = S1_ARCH_A_DIR / "model_A_metadata.json"
S1_A_STATE_DICT_PATH = S1_ARCH_A_DIR / "model_A_state_dict.pt"

if not S1_A_METADATA_PATH.is_file():
    raise FileNotFoundError(f"Missing metadata: {S1_A_METADATA_PATH}")
if not S1_A_STATE_DICT_PATH.is_file():
    raise FileNotFoundError(f"Missing state dict: {S1_A_STATE_DICT_PATH}")

with open(S1_A_METADATA_PATH, "r", encoding="utf-8") as f:
    S1_A_METADATA = json.load(f)

S1_A_CONFIG = S1_A_METADATA["model_config"]
S1_A_EXPECTED_PARAMS = int(S1_A_METADATA["parameter_count"])

print("=" * 68)
print("S1 REDO — ARCHITECTURE A CONFIGURATION")
print("=" * 68)
print(json.dumps(S1_A_CONFIG, indent=2))
print(f"Metadata parameter count: {S1_A_EXPECTED_PARAMS:,}")
print(f"State-dict file size: {S1_A_STATE_DICT_PATH.stat().st_size / 1024**2:.1f} MiB")

# mmap avoids eagerly copying the entire checkpoint into another CPU allocation.
try:
    S1_A_LOADED_OBJECT = torch.load(
        S1_A_STATE_DICT_PATH,
        map_location="cpu",
        weights_only=True,
        mmap=True,
    )
except Exception as exc:
    raise RuntimeError(
        "Could not load model_A_state_dict.pt with CPU memory mapping. "
        f"Underlying error: {type(exc).__name__}: {exc}"
    ) from exc

# Support either a direct state dict or a wrapper around one.
S1_A_STATE_DICT = None

if isinstance(S1_A_LOADED_OBJECT, dict):
    if S1_A_LOADED_OBJECT and all(
        torch.is_tensor(v) for v in S1_A_LOADED_OBJECT.values()
    ):
        S1_A_STATE_DICT = S1_A_LOADED_OBJECT
    else:
        for key in ("model_state_dict", "state_dict", "model"):
            value = S1_A_LOADED_OBJECT.get(key)
            if isinstance(value, dict) and value and all(
                torch.is_tensor(v) for v in value.values()
            ):
                S1_A_STATE_DICT = value
                print("State dict wrapper key:", key)
                break

if S1_A_STATE_DICT is None:
    print("Loaded object type:", type(S1_A_LOADED_OBJECT).__name__)
    if isinstance(S1_A_LOADED_OBJECT, dict):
        print("Top-level keys:", list(S1_A_LOADED_OBJECT.keys())[:30])
    raise TypeError(
        "Could not identify a tensor-only model state dict. "
        "Review the displayed checkpoint structure before proceeding."
    )

S1_A_TENSOR_COUNT = len(S1_A_STATE_DICT)
S1_A_STATE_PARAM_COUNT = sum(
    tensor.numel()
    for tensor in S1_A_STATE_DICT.values()
    if tensor.is_floating_point()
)

print("\n" + "=" * 68)
print("PRETRAINED STATE-DICT SUMMARY")
print("=" * 68)
print("Tensor entries:", S1_A_TENSOR_COUNT)
print(f"Floating-point parameters represented: {S1_A_STATE_PARAM_COUNT:,}")
print("Metadata parameters:", f"{S1_A_EXPECTED_PARAMS:,}")
print("Parameter-count match:", S1_A_STATE_PARAM_COUNT == S1_A_EXPECTED_PARAMS)

print("\nFirst 45 tensor names and shapes:")
for i, (name, tensor) in enumerate(S1_A_STATE_DICT.items()):
    if i >= 45:
        break
    print(f"{name:60s} {str(tuple(tensor.shape)):24s} {tensor.dtype}")

print("\nPotential embedding / output / position tensors:")
for name, tensor in S1_A_STATE_DICT.items():
    low = name.lower()
    if any(term in low for term in (
        "embed", "token", "position", "pos_emb", "lm_head",
        "output", "unembed", "wte", "wpe"
    )):
        print(f"{name:60s} {tuple(tensor.shape)}")

print("\nCell 2 finished. No training has been performed.")

S1 REDO — ARCHITECTURE A CONFIGURATION
{
  "vocab_size": 50000,
  "context_length": 2048,
  "d_model": 768,
  "n_layers": 12,
  "n_heads": 12,
  "d_ff": 3072,
  "dropout": 0.1,
  "activation": "gelu",
  "norm_type": "layernorm",
  "pre_norm": true,
  "position_type": "learned",
  "tie_embeddings": true,
  "use_bias": true
}
Metadata parameter count: 125,028,864
State-dict file size: 477.0 MiB

PRETRAINED STATE-DICT SUMMARY
Tensor entries: 197
Floating-point parameters represented: 163,428,864
Metadata parameters: 125,028,864
Parameter-count match: False

First 45 tensor names and shapes:
output_weight                                                (50000, 768)             torch.float32
token_embedding.weight                                       (50000, 768)             torch.float32
position_embedding.weight                                    (2048, 768)              torch.float32
blocks.0.ln_1.weight                                         (768,)                   torch.float32
block

In [3]:
# ============================================================
# S1 REDO — CELL 3
# Purpose:
#   1. Verify whether output and token embeddings are tied.
#   2. Inspect every parameter name outside the Transformer blocks.
#   3. Confirm that all 12 Transformer blocks are present.
#   4. Establish the exact key structure for strict model loading.
# ============================================================

import re
import torch

assert "S1_A_STATE_DICT" in globals(), "Run Cells 1 and 2 first."

S1_A_KEYS = list(S1_A_STATE_DICT.keys())

# ---------- Verify tied embedding tensors ----------
S1_TOKEN_EMBED = S1_A_STATE_DICT.get("token_embedding.weight")
S1_OUTPUT_EMBED = S1_A_STATE_DICT.get("output_weight")

print("=" * 68)
print("S1 REDO — EMBEDDING TIE VERIFICATION")
print("=" * 68)

if S1_TOKEN_EMBED is None or S1_OUTPUT_EMBED is None:
    print("Could not find both token_embedding.weight and output_weight.")
    print("Token embedding found:", S1_TOKEN_EMBED is not None)
    print("Output embedding found:", S1_OUTPUT_EMBED is not None)
else:
    print("Token embedding shape:", tuple(S1_TOKEN_EMBED.shape))
    print("Output weight shape:   ", tuple(S1_OUTPUT_EMBED.shape))
    print("Exact tensor equality: ", torch.equal(S1_TOKEN_EMBED, S1_OUTPUT_EMBED))
    print(
        "Maximum absolute difference:",
        float((S1_TOKEN_EMBED - S1_OUTPUT_EMBED).abs().max().item())
    )

# ---------- Confirm Transformer block indices ----------
S1_BLOCK_INDICES = sorted({
    int(match.group(1))
    for key in S1_A_KEYS
    if (match := re.match(r"blocks\.(\d+)\.", key))
})

print("\n" + "=" * 68)
print("TRANSFORMER BLOCK STRUCTURE")
print("=" * 68)
print("Block indices:", S1_BLOCK_INDICES)
print("Number of blocks:", len(S1_BLOCK_INDICES))
print("Expected blocks:", S1_A_CONFIG["n_layers"])
print("All expected blocks present:", S1_BLOCK_INDICES == list(range(S1_A_CONFIG["n_layers"])))

# ---------- Show all non-block tensor names ----------
S1_NONBLOCK_KEYS = [
    key for key in S1_A_KEYS
    if not re.match(r"blocks\.\d+\.", key)
]

print("\nNon-block state-dict entries:")
for key in S1_NONBLOCK_KEYS:
    tensor = S1_A_STATE_DICT[key]
    print(f"  {key:45s} shape={tuple(tensor.shape)} dtype={tensor.dtype}")

# ---------- Summarize each block's modules ----------
print("\nModule names in each Transformer block:")
for block_idx in S1_BLOCK_INDICES:
    prefix = f"blocks.{block_idx}."
    modules = sorted({
        ".".join(key[len(prefix):].split(".")[:2])
        for key in S1_A_KEYS
        if key.startswith(prefix)
    })
    print(f"  Block {block_idx:02d}: {', '.join(modules)}")

# ---------- Detect missing/unexpected patterns ----------
S1_EXPECTED_HEADS = [
    "token_embedding.weight",
    "position_embedding.weight",
    "output_weight",
]

print("\nRequired top-level weights present:")
for key in S1_EXPECTED_HEADS:
    print(f"  {key}: {key in S1_A_STATE_DICT}")

print("\nCell 3 finished. This cell only inspected the checkpoint.")

S1 REDO — EMBEDDING TIE VERIFICATION
Token embedding shape: (50000, 768)
Output weight shape:    (50000, 768)
Exact tensor equality:  True
Maximum absolute difference: 0.0

TRANSFORMER BLOCK STRUCTURE
Block indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Number of blocks: 12
Expected blocks: 12
All expected blocks present: True

Non-block state-dict entries:
  output_weight                                 shape=(50000, 768) dtype=torch.float32
  token_embedding.weight                        shape=(50000, 768) dtype=torch.float32
  position_embedding.weight                     shape=(2048, 768) dtype=torch.float32
  final_norm.weight                             shape=(768,) dtype=torch.float32
  final_norm.bias                               shape=(768,) dtype=torch.float32

Module names in each Transformer block:
  Block 00: attention.k_proj, attention.out_proj, attention.q_proj, attention.v_proj, feed_forward.fc1, feed_forward.fc2, ln_1.bias, ln_1.weight, ln_2.bias, ln_2.weight
  Block

In [4]:
# ============================================================
# S1 REDO — CELL 4
# Purpose:
#   1. Reconstruct the Phase 1 decoder-only Transformer.
#   2. Preserve the verified tied input/output embeddings.
#   3. Strictly load all pretrained Architecture A weights.
#   4. Verify parameter count and run a forward-pass smoke test.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

assert "S1_A_STATE_DICT" in globals(), "Run Cells 1–3 first."
assert "S1_A_CONFIG" in globals(), "Architecture A config is missing."

# ---------- Configuration ----------
S1_VOCAB_SIZE = int(S1_A_CONFIG["vocab_size"])
S1_CONTEXT_LENGTH = int(S1_A_CONFIG["context_length"])
S1_D_MODEL = int(S1_A_CONFIG["d_model"])
S1_N_LAYERS = int(S1_A_CONFIG["n_layers"])
S1_N_HEADS = int(S1_A_CONFIG["n_heads"])
S1_D_FF = int(S1_A_CONFIG["d_ff"])
S1_DROPOUT = float(S1_A_CONFIG["dropout"])

assert S1_D_MODEL % S1_N_HEADS == 0
S1_HEAD_DIM = S1_D_MODEL // S1_N_HEADS


# ---------- Multi-head causal self-attention ----------
class S1CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        d_model = int(config["d_model"])
        n_heads = int(config["n_heads"])

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=config["use_bias"])
        self.k_proj = nn.Linear(d_model, d_model, bias=config["use_bias"])
        self.v_proj = nn.Linear(d_model, d_model, bias=config["use_bias"])
        self.out_proj = nn.Linear(d_model, d_model, bias=config["use_bias"])
        self.dropout = nn.Dropout(float(config["dropout"]))

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        q = self.q_proj(x).view(
            batch_size, seq_len, self.n_heads, self.head_dim
        ).transpose(1, 2)

        k = self.k_proj(x).view(
            batch_size, seq_len, self.n_heads, self.head_dim
        ).transpose(1, 2)

        v = self.v_proj(x).view(
            batch_size, seq_len, self.n_heads, self.head_dim
        ).transpose(1, 2)

        # Flash/memory-efficient attention may be used by PyTorch SDPA.
        # The causal mask prevents a token from attending to future tokens.
        attention_output = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=True,
        )

        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(
            batch_size, seq_len, d_model
        )

        return self.dropout(self.out_proj(attention_output))


# ---------- Feed-forward network ----------
class S1FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(
            int(config["d_model"]),
            int(config["d_ff"]),
            bias=config["use_bias"],
        )
        self.fc2 = nn.Linear(
            int(config["d_ff"]),
            int(config["d_model"]),
            bias=config["use_bias"],
        )
        self.dropout = nn.Dropout(float(config["dropout"]))

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return self.dropout(x)


# ---------- Pre-normalized Transformer block ----------
class S1TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        d_model = int(config["d_model"])

        self.ln_1 = nn.LayerNorm(d_model)
        self.attention = S1CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(d_model)
        self.feed_forward = S1FeedForward(config)

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        x = x + self.feed_forward(self.ln_2(x))
        return x


# ---------- Architecture A backbone ----------
class S1ArchitectureA(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.vocab_size = int(config["vocab_size"])
        self.context_length = int(config["context_length"])
        d_model = int(config["d_model"])

        self.token_embedding = nn.Embedding(
            self.vocab_size, d_model
        )
        self.position_embedding = nn.Embedding(
            self.context_length, d_model
        )

        self.blocks = nn.ModuleList([
            S1TransformerBlock(config)
            for _ in range(int(config["n_layers"]))
        ])

        self.final_norm = nn.LayerNorm(d_model)

        # Exact parameter tying, verified against the Phase 1 checkpoint.
        self.output_weight = self.token_embedding.weight

        self.embedding_dropout = nn.Dropout(float(config["dropout"]))

    def forward(self, input_ids):
        if input_ids.ndim != 2:
            raise ValueError(
                f"Expected input_ids shaped [batch, sequence], "
                f"received {tuple(input_ids.shape)}"
            )

        batch_size, seq_len = input_ids.shape
        if seq_len > self.context_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"context length {self.context_length}"
            )

        positions = torch.arange(
            seq_len, device=input_ids.device
        ).unsqueeze(0).expand(batch_size, -1)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )
        x = self.embedding_dropout(x)

        for block in self.blocks:
            x = block(x)

        return self.final_norm(x)


# ---------- Instantiate and strictly load the pretrained weights ----------
S1_A_MODEL = S1ArchitectureA(S1_A_CONFIG)

# Strict loading is intentional: mismatched keys must fail loudly.
S1_A_LOAD_RESULT = S1_A_MODEL.load_state_dict(
    S1_A_STATE_DICT,
    strict=True,
)

S1_A_MODEL = S1_A_MODEL.to(S1_DEVICE)
S1_A_MODEL.eval()

S1_A_UNIQUE_PARAM_COUNT = sum(
    p.numel() for p in S1_A_MODEL.parameters()
)
S1_A_TRAINABLE_PARAM_COUNT = sum(
    p.numel() for p in S1_A_MODEL.parameters() if p.requires_grad
)

print("=" * 68)
print("S1 REDO — ARCHITECTURE A LOADED")
print("=" * 68)
print("Strict load result:", S1_A_LOAD_RESULT)
print(f"Unique parameters: {S1_A_UNIQUE_PARAM_COUNT:,}")
print(f"Metadata parameters: {S1_A_EXPECTED_PARAMS:,}")
print(
    "Parameter-count match:",
    S1_A_UNIQUE_PARAM_COUNT == S1_A_EXPECTED_PARAMS
)
print("Transformer blocks:", len(S1_A_MODEL.blocks))
print(
    "Input/output embedding tied:",
    S1_A_MODEL.token_embedding.weight.data_ptr()
    == S1_A_MODEL.output_weight.data_ptr()
)

if S1_A_UNIQUE_PARAM_COUNT != S1_A_EXPECTED_PARAMS:
    raise RuntimeError(
        "Reconstructed parameter count differs from Architecture A metadata."
    )

# ---------- Forward-pass smoke test ----------
S1_SMOKE_IDS = torch.randint(
    low=0,
    high=S1_VOCAB_SIZE,
    size=(1, 32),
    device=S1_DEVICE,
)

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(S1_DEVICE.type == "cuda"),
    ):
        S1_SMOKE_OUTPUT = S1_A_MODEL(S1_SMOKE_IDS)

assert S1_SMOKE_OUTPUT.shape == (1, 32, S1_D_MODEL)
assert torch.isfinite(S1_SMOKE_OUTPUT).all().item()

print("Forward-pass input shape: ", tuple(S1_SMOKE_IDS.shape))
print("Forward-pass output shape:", tuple(S1_SMOKE_OUTPUT.shape))
print("Output finite:", torch.isfinite(S1_SMOKE_OUTPUT).all().item())
print("\nCell 4 finished. Pretrained weights remain intact; no fine-tuning yet.")

S1 REDO — ARCHITECTURE A LOADED
Strict load result: <All keys matched successfully>
Unique parameters: 125,028,864
Metadata parameters: 125,028,864
Parameter-count match: True
Transformer blocks: 12
Input/output embedding tied: True
Forward-pass input shape:  (1, 32)
Forward-pass output shape: (1, 32, 768)
Output finite: True

Cell 4 finished. Pretrained weights remain intact; no fine-tuning yet.


In [5]:
# ============================================================
# S1 REDO — CELL 5
# Purpose:
#   1. Load the Phase 1 tokenizer.
#   2. Inspect the official gold-annotation structure.
#   3. Verify token offsets map to the original document text.
#   4. Count invalid or unsupported annotations before training.
# ============================================================

from tokenizers import Tokenizer
from collections import Counter

S1_TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/rohitbatra2004/"
    "cs781-phase1-tokenized-corpus/tokenizer/tokenizer.json"
)

if not S1_TOKENIZER_PATH.is_file():
    raise FileNotFoundError(f"Tokenizer not found: {S1_TOKENIZER_PATH}")

S1_TOKENIZER = Tokenizer.from_file(str(S1_TOKENIZER_PATH))
S1_TOKENIZER.enable_truncation(max_length=S1_CONTEXT_LENGTH)

print("=" * 68)
print("S1 REDO — ANNOTATION AND TOKENIZER CHECK")
print("=" * 68)
print("Tokenizer vocabulary:", S1_TOKENIZER.get_vocab_size())
print("Architecture vocabulary:", S1_VOCAB_SIZE)

if S1_TOKENIZER.get_vocab_size() != S1_VOCAB_SIZE:
    raise ValueError("Tokenizer vocabulary size does not match Architecture A.")

# ---------- Inspect annotation structure on one real training document ----------
S1_SAMPLE_DOC_ID = next(iter(S1_FOLD1_TEXT))
S1_SAMPLE_TEXT = S1_FOLD1_TEXT[S1_SAMPLE_DOC_ID]
S1_SAMPLE_GOLD = S1_FOLD1_GOLD[S1_SAMPLE_DOC_ID]

print("\nSample document ID:", S1_SAMPLE_DOC_ID)
print("Sample text length:", len(S1_SAMPLE_TEXT))
print("Gold annotation container:", type(S1_SAMPLE_GOLD).__name__)

if isinstance(S1_SAMPLE_GOLD, dict):
    print("Gold keys:", list(S1_SAMPLE_GOLD.keys())[:20])
    for key, value in list(S1_SAMPLE_GOLD.items())[:2]:
        print(f"  {key!r}: type={type(value).__name__}, sample={str(value)[:500]}")
elif isinstance(S1_SAMPLE_GOLD, list):
    print("Gold annotation count:", len(S1_SAMPLE_GOLD))
    for item in S1_SAMPLE_GOLD[:3]:
        print("  ", repr(item)[:700])
else:
    print("Unexpected gold object:", repr(S1_SAMPLE_GOLD)[:1000])

# ---------- Verify tokenizer output and offset convention ----------
S1_SAMPLE_ENCODING = S1_TOKENIZER.encode(S1_SAMPLE_TEXT)
S1_SAMPLE_IDS = S1_SAMPLE_ENCODING.ids
S1_SAMPLE_OFFSETS = S1_SAMPLE_ENCODING.offsets
S1_SAMPLE_TOKENS = S1_SAMPLE_ENCODING.tokens

print("\nTokenizer output:")
print("Token count:", len(S1_SAMPLE_IDS))
print("First 20 token IDs:", S1_SAMPLE_IDS[:20])
print("First 20 tokens:", S1_SAMPLE_TOKENS[:20])
print("First 20 offsets:", S1_SAMPLE_OFFSETS[:20])

print("\nOffset spot checks:")
S1_OFFSET_CHECKS = 0
for token, (start, end) in zip(
    S1_SAMPLE_TOKENS, S1_SAMPLE_OFFSETS
):
    if start < end and 0 <= start < end <= len(S1_SAMPLE_TEXT):
        print(
            f"  offset=({start},{end}) | "
            f"token={token!r} | text={S1_SAMPLE_TEXT[start:end]!r}"
        )
        S1_OFFSET_CHECKS += 1
    if S1_OFFSET_CHECKS >= 5:
        break

if S1_OFFSET_CHECKS == 0:
    raise ValueError("Tokenizer did not produce any valid character offsets.")

# ---------- Count annotation types without touching Fold 3 gold ----------
S1_ANNOTATION_TYPES = Counter()
S1_FOLD1_GOLD_COUNTS = []
S1_FOLD2_GOLD_COUNTS = []

for doc_id, annotations in S1_FOLD1_GOLD.items():
    S1_FOLD1_GOLD_COUNTS.append(
        len(annotations) if isinstance(annotations, (list, dict)) else -1
    )
    S1_ANNOTATION_TYPES[type(annotations).__name__] += 1

for doc_id, annotations in S1_FOLD2_GOLD.items():
    S1_FOLD2_GOLD_COUNTS.append(
        len(annotations) if isinstance(annotations, (list, dict)) else -1
    )
    S1_ANNOTATION_TYPES[type(annotations).__name__] += 1

print("\nGold-container types across training folds:", dict(S1_ANNOTATION_TYPES))
print(
    "Fold 1 annotations per document — min/median/max:",
    min(S1_FOLD1_GOLD_COUNTS),
    int(np.median(S1_FOLD1_GOLD_COUNTS)),
    max(S1_FOLD1_GOLD_COUNTS),
)
print(
    "Fold 2 annotations per document — min/median/max:",
    min(S1_FOLD2_GOLD_COUNTS),
    int(np.median(S1_FOLD2_GOLD_COUNTS)),
    max(S1_FOLD2_GOLD_COUNTS),
)
print("Cell 5 finished. No training performed.")

S1 REDO — ANNOTATION AND TOKENIZER CHECK
Tokenizer vocabulary: 50000
Architecture vocabulary: 50000

Sample document ID: 115651329
Sample text length: 18616
Gold annotation container: list
Gold annotation count: 104
   [137, 153, 'RESP', 'STATE OF HARYANA']
   [252, 261, 'DATE', '19.8.2011']
   [276, 308, 'COURT', 'High Court of Punjab and Haryana']

Tokenizer output:
Token count: 2048
First 20 token IDs: [9269, 2720, 2497, 8361, 4398, 1912, 7359, 2060, 5759, 5387, 2060, 3240, 3188, 17, 8734, 18, 3688, 44005, 40, 20617]
First 20 tokens: ['ĠREPORTABLE', 'ĠIN', 'ĠTHE', 'ĠSUPREME', 'ĠCOURT', 'ĠOF', 'ĠINDIA', 'ĠCRIMINAL', 'ĠAPPELLATE', 'ĠJURISDICTION', 'ĠCRIMINAL', 'ĠAPPEAL', 'ĠNO', '.', 'Ġ92', '/', '2015', 'ĠJAG', 'E', 'ĠRAM']
First 20 offsets: [(0, 10), (10, 13), (13, 17), (17, 25), (25, 31), (31, 34), (34, 40), (40, 49), (49, 59), (59, 72), (72, 81), (81, 88), (88, 91), (91, 92), (92, 95), (95, 96), (96, 100), (100, 104), (104, 105), (105, 109)]

Offset spot checks:
  offset=(0,10) | to

In [6]:
# ============================================================
# S1 REDO — CELL 6
# Purpose:
#   1. Disable tokenizer truncation before processing documents.
#   2. Parse the official [start, end, label, entity_text] format.
#   3. Verify span bounds, labels, and annotation text.
#   4. Measure character-boundary alignment with tokenizer offsets.
#   5. Create a reproducible document-level train/validation split.
# ============================================================

from collections import Counter
from sklearn.model_selection import train_test_split
import numpy as np

assert "S1_TOKENIZER" in globals()
assert "S1_FOLD1_TEXT" in globals()
assert "S1_FOLD1_GOLD" in globals()
assert "S1_FOLD2_TEXT" in globals()

# Cell 5 enabled 2048-token truncation. Disable it for all NER documents.
S1_TOKENIZER.no_truncation()

# ---------- Merge both annotated folds without ID collisions ----------
S1_ALL_TEXTS = {}
S1_ALL_GOLD_RAW = {}

for source_texts, source_gold, fold_name in [
    (S1_FOLD1_TEXT, S1_FOLD1_GOLD, "fold1"),
    (S1_FOLD2_TEXT, S1_FOLD2_GOLD, "fold2"),
]:
    source_texts = {str(k): v for k, v in source_texts.items()}
    source_gold = {str(k): v for k, v in source_gold.items()}

    if set(source_texts) != set(source_gold):
        raise ValueError(f"{fold_name}: text/gold document IDs do not match.")

    duplicate_ids = set(S1_ALL_TEXTS) & set(source_texts)
    if duplicate_ids:
        raise ValueError(
            f"Document IDs overlap across folds: {sorted(duplicate_ids)[:5]}"
        )

    S1_ALL_TEXTS.update(source_texts)
    S1_ALL_GOLD_RAW.update(source_gold)

S1_LABEL_TO_ID = {label: i for i, label in enumerate(S1_LABELS)}
S1_ID_TO_LABEL = {i: label for label, i in S1_LABEL_TO_ID.items()}
S1_NUM_LABELS = len(S1_LABELS)

# ---------- Parse the dataset's entity annotations ----------
def s1_parse_annotation(item):
    """Return (start, end, label, optional_text) for supported annotation rows."""
    if isinstance(item, dict):
        start = next(
            (item[k] for k in ("start", "start_char", "start_character")
             if k in item), None
        )
        end = next(
            (item[k] for k in ("end", "end_char", "end_character")
             if k in item), None
        )
        label = next(
            (item[k] for k in ("label", "entity", "type")
             if k in item), None
        )
        entity_text = next(
            (item[k] for k in ("text", "entity_text", "surface")
             if k in item), None
        )
    elif isinstance(item, (list, tuple)) and len(item) >= 3:
        start, end, label = item[:3]
        entity_text = item[3] if len(item) >= 4 else None
    else:
        raise ValueError(f"Unsupported annotation record: {repr(item)[:250]}")

    if isinstance(start, bool) or isinstance(end, bool):
        raise ValueError(f"Boolean span offsets are invalid: {item!r}")

    try:
        start, end = int(start), int(end)
    except (ValueError, TypeError) as exc:
        raise ValueError(f"Non-integer annotation offsets: {item!r}") from exc

    label = str(label).strip()
    return start, end, label, entity_text


S1_SPANS_BY_DOC = {}
S1_ANNOTATION_TEXT_MISMATCHES = []
S1_INVALID_ANNOTATIONS = []
S1_LABEL_COUNTS = Counter()

for doc_id, text in S1_ALL_TEXTS.items():
    parsed_spans = []

    for annotation_index, item in enumerate(S1_ALL_GOLD_RAW[doc_id]):
        try:
            start, end, label, supplied_text = s1_parse_annotation(item)
        except Exception as exc:
            S1_INVALID_ANNOTATIONS.append(
                (doc_id, annotation_index, str(exc))
            )
            continue

        if (
            not (0 <= start < end <= len(text))
            or label not in S1_LABEL_TO_ID
        ):
            S1_INVALID_ANNOTATIONS.append(
                (doc_id, annotation_index, start, end, label, len(text))
            )
            continue

        actual_text = text[start:end]
        if supplied_text is not None and str(supplied_text) != actual_text:
            S1_ANNOTATION_TEXT_MISMATCHES.append(
                (doc_id, annotation_index, start, end,
                 str(supplied_text)[:100], actual_text[:100])
            )

        parsed_spans.append((start, end, label))
        S1_LABEL_COUNTS[label] += 1

    S1_SPANS_BY_DOC[doc_id] = parsed_spans

if S1_INVALID_ANNOTATIONS:
    raise ValueError(
        f"Found {len(S1_INVALID_ANNOTATIONS)} invalid annotation records. "
        f"Examples: {S1_INVALID_ANNOTATIONS[:5]}"
    )

# Annotation text mismatches are reported separately. They are not silently
# deleted: the next step will use the official character offsets.
print("=" * 68)
print("S1 REDO — GOLD ANNOTATION VALIDATION")
print("=" * 68)
print("Documents:", len(S1_ALL_TEXTS))
print("Gold spans:", sum(len(v) for v in S1_SPANS_BY_DOC.values()))
print("Invalid offsets/labels:", len(S1_INVALID_ANNOTATIONS))
print("Supplied entity-text mismatches:", len(S1_ANNOTATION_TEXT_MISMATCHES))
if S1_ANNOTATION_TEXT_MISMATCHES:
    print("Mismatch examples:", S1_ANNOTATION_TEXT_MISMATCHES[:3])

print("\nGold spans per label:")
for label in S1_LABELS:
    print(f"  {label:12s} {S1_LABEL_COUNTS[label]:5d}")

# ---------- Tokenize every complete document without truncation ----------
S1_ENCODED_DOCS = {}
S1_OFFSET_ALIGNMENT = Counter()
S1_TOKEN_COUNTS = []

for doc_id, text in S1_ALL_TEXTS.items():
    encoding = S1_TOKENIZER.encode(text)
    offsets = encoding.offsets

    if len(encoding.ids) != len(offsets):
        raise RuntimeError(f"{doc_id}: token IDs and offsets have different lengths.")

    S1_ENCODED_DOCS[doc_id] = {
        "ids": encoding.ids,
        "offsets": offsets,
    }
    S1_TOKEN_COUNTS.append(len(encoding.ids))

    valid_offsets = [
        (int(start), int(end))
        for start, end in offsets
        if 0 <= start < end <= len(text)
    ]
    token_starts = {start for start, _ in valid_offsets}
    token_ends = {end for _, end in valid_offsets}

    for start, end, label in S1_SPANS_BY_DOC[doc_id]:
        S1_OFFSET_ALIGNMENT["total"] += 1
        start_aligned = start in token_starts
        end_aligned = end in token_ends
        S1_OFFSET_ALIGNMENT["start_aligned"] += int(start_aligned)
        S1_OFFSET_ALIGNMENT["end_aligned"] += int(end_aligned)
        S1_OFFSET_ALIGNMENT["both_aligned"] += int(
            start_aligned and end_aligned
        )

        # Character-level prediction will retain the official offsets even if
        # a gold span starts or ends inside a tokenizer token.

S1_TOTAL_SPANS = S1_OFFSET_ALIGNMENT["total"]
print("\n" + "=" * 68)
print("S1 REDO — FULL-DOCUMENT TOKENIZATION")
print("=" * 68)
print("Truncation enabled:", False)
print("Documents tokenized:", len(S1_ENCODED_DOCS))
print(
    "Tokens per document (min / median / max):",
    min(S1_TOKEN_COUNTS),
    int(np.median(S1_TOKEN_COUNTS)),
    max(S1_TOKEN_COUNTS),
)
print(f"Gold start boundaries aligned: {S1_OFFSET_ALIGNMENT['start_aligned']}/{S1_TOTAL_SPANS}")
print(f"Gold end boundaries aligned:   {S1_OFFSET_ALIGNMENT['end_aligned']}/{S1_TOTAL_SPANS}")
print(f"Both boundaries aligned:       {S1_OFFSET_ALIGNMENT['both_aligned']}/{S1_TOTAL_SPANS}")

# ---------- Reproducible document-level validation split ----------
S1_ALL_DOC_IDS = sorted(S1_ALL_TEXTS.keys())
S1_TRAIN_DOC_IDS, S1_VAL_DOC_IDS = train_test_split(
    S1_ALL_DOC_IDS,
    test_size=0.20,
    random_state=S1_SEED,
    shuffle=True,
)

S1_TRAIN_DOC_IDS = sorted(S1_TRAIN_DOC_IDS)
S1_VAL_DOC_IDS = sorted(S1_VAL_DOC_IDS)

S1_TRAIN_SPAN_COUNT = sum(len(S1_SPANS_BY_DOC[x]) for x in S1_TRAIN_DOC_IDS)
S1_VAL_SPAN_COUNT = sum(len(S1_SPANS_BY_DOC[x]) for x in S1_VAL_DOC_IDS)

assert not (set(S1_TRAIN_DOC_IDS) & set(S1_VAL_DOC_IDS))
assert len(S1_TRAIN_DOC_IDS) + len(S1_VAL_DOC_IDS) == len(S1_ALL_TEXTS)

print("\n" + "=" * 68)
print("S1 REDO — DOCUMENT-LEVEL SPLIT")
print("=" * 68)
print("Training documents:", len(S1_TRAIN_DOC_IDS))
print("Validation documents:", len(S1_VAL_DOC_IDS))
print("Training gold spans:", S1_TRAIN_SPAN_COUNT)
print("Validation gold spans:", S1_VAL_SPAN_COUNT)
print("Document leakage:", bool(set(S1_TRAIN_DOC_IDS) & set(S1_VAL_DOC_IDS)))
print("Cell 6 finished. No fine-tuning performed.")

S1 REDO — GOLD ANNOTATION VALIDATION
Documents: 70
Gold spans: 8393
Invalid offsets/labels: 0
Supplied entity-text mismatches: 0

Gold spans per label:
  APP            407
  RESP           381
  A.COUNSEL      195
  R.COUNSEL      184
  JUDGE          245
  WIT            187
  AUTH          1003
  COURT         1013
  STAT          1260
  PREC          1183
  DATE          1575
  CASENO         760

S1 REDO — FULL-DOCUMENT TOKENIZATION
Truncation enabled: False
Documents tokenized: 70
Tokens per document (min / median / max): 2108 5483 23559
Gold start boundaries aligned: 317/8393
Gold end boundaries aligned:   8241/8393
Both boundaries aligned:       307/8393

S1 REDO — DOCUMENT-LEVEL SPLIT
Training documents: 56
Validation documents: 14
Training gold spans: 6532
Validation gold spans: 1861
Document leakage: False
Cell 6 finished. No fine-tuning performed.


In [7]:
# ============================================================
# S1 REDO — CELL 7
# Purpose:
#   1. Build a character vocabulary using training documents only.
#   2. Split long legal documents into overlapping token windows.
#   3. Map every character to its local Transformer token.
#   4. Create character-level start/end targets for all 12 labels.
#   5. Verify gold-span window coverage before training.
# ============================================================

import numpy as np
from collections import Counter

assert "S1_ENCODED_DOCS" in globals()
assert "S1_TRAIN_DOC_IDS" in globals()
assert "S1_VAL_DOC_IDS" in globals()
assert "S1_SPANS_BY_DOC" in globals()

S1_WINDOW_TOKENS = 512
S1_WINDOW_STRIDE = 384

# ---------- Character vocabulary: training text only ----------
S1_CHAR_VOCAB = sorted({
    char
    for doc_id in S1_TRAIN_DOC_IDS
    for char in S1_ALL_TEXTS[doc_id]
})

S1_CHAR_PAD_ID = 0
S1_CHAR_UNK_ID = 1
S1_CHAR_TO_ID = {
    char: idx + 2 for idx, char in enumerate(S1_CHAR_VOCAB)
}
S1_CHAR_VOCAB_SIZE = len(S1_CHAR_TO_ID) + 2

print("=" * 68)
print("S1 REDO — WINDOW AND CHARACTER-TARGET SETUP")
print("=" * 68)
print("Window tokens:", S1_WINDOW_TOKENS)
print("Window stride:", S1_WINDOW_STRIDE)
print("Character vocabulary size:", S1_CHAR_VOCAB_SIZE)

def s1_make_window_starts(token_count, window_size, stride):
    """Create overlapping windows, aligning the final window to document end."""
    if token_count <= 0:
        return []
    if token_count <= window_size:
        return [0]

    starts = [0]
    while starts[-1] + window_size < token_count:
        next_start = starts[-1] + stride

        if next_start + window_size >= token_count:
            next_start = token_count - window_size

        if next_start <= starts[-1]:
            break

        starts.append(next_start)

    return starts


def s1_char_to_local_token(offsets, char_start, char_end):
    """
    Map every character in [char_start, char_end) to a local token index.
    Characters in gaps are mapped to the nearest available preceding token;
    leading uncovered characters use the first available token.
    """
    char_length = char_end - char_start
    if char_length <= 0:
        raise ValueError(
            f"Invalid character window: ({char_start}, {char_end})"
        )

    mapping = np.full(char_length, -1, dtype=np.int32)

    for local_idx, (global_start, global_end) in enumerate(offsets):
        start = max(int(global_start), char_start)
        end = min(int(global_end), char_end)

        if start < end:
            mapping[start - char_start:end - char_start] = local_idx

    valid_positions = np.flatnonzero(mapping >= 0)
    if len(valid_positions) == 0:
        raise ValueError(
            f"No valid token offsets cover character window "
            f"({char_start}, {char_end})"
        )

    # Fill any leading uncovered characters using the first token with an offset.
    first_valid = int(valid_positions[0])
    mapping[:first_valid] = mapping[first_valid]

    # Fill gaps and trailing characters with the most recent valid token.
    last_token = int(mapping[first_valid])
    for position in range(first_valid, char_length):
        if mapping[position] >= 0:
            last_token = int(mapping[position])
        else:
            mapping[position] = last_token

    if np.any(mapping < 0) or np.any(mapping >= len(offsets)):
        raise RuntimeError("Character-to-token mapping contains invalid indices.")

    return mapping


def s1_build_doc_windows(doc_id, include_labels=True):
    """
    Return overlapping windows with token inputs and character-level targets.
    Character offsets are always in the original document coordinate system.
    """
    text = S1_ALL_TEXTS[doc_id]
    encoded = S1_ENCODED_DOCS[doc_id]
    token_ids = encoded["ids"]
    offsets = encoded["offsets"]

    if not token_ids or len(token_ids) != len(offsets):
        raise ValueError(f"{doc_id}: invalid or empty tokenizer output.")

    starts = s1_make_window_starts(
        len(token_ids), S1_WINDOW_TOKENS, S1_WINDOW_STRIDE
    )

    spans = S1_SPANS_BY_DOC[doc_id] if include_labels else []
    windows = []
    covered_spans = set()

    for token_start in starts:
        token_end = min(token_start + S1_WINDOW_TOKENS, len(token_ids))
        local_ids = token_ids[token_start:token_end]
        local_offsets = offsets[token_start:token_end]

        if not local_ids:
            continue

        # The first and final windows include document-edge characters.
        char_start = (
            0 if token_start == 0 else int(local_offsets[0][0])
        )
        char_end = (
            len(text) if token_end == len(token_ids)
            else int(local_offsets[-1][1])
        )
        char_start = max(0, char_start)
        char_end = min(len(text), char_end)

        if char_end <= char_start:
            raise ValueError(
                f"{doc_id}: invalid window character offsets "
                f"({char_start}, {char_end})"
            )

        local_char_to_token = s1_char_to_local_token(
            local_offsets, char_start, char_end
        )
        char_text = text[char_start:char_end]

        char_ids = np.fromiter(
            (
                S1_CHAR_TO_ID.get(char, S1_CHAR_UNK_ID)
                for char in char_text
            ),
            dtype=np.int32,
            count=len(char_text),
        )

        start_targets = np.zeros(
            (len(char_text), S1_NUM_LABELS), dtype=np.float32
        )
        end_targets = np.zeros_like(start_targets)

        if include_labels:
            for span in spans:
                start, end, label = span
                if char_start <= start < end <= char_end:
                    label_id = S1_LABEL_TO_ID[label]
                    start_targets[start - char_start, label_id] = 1.0
                    # End targets mark the final included character.
                    end_targets[end - char_start - 1, label_id] = 1.0
                    covered_spans.add(span)

        windows.append({
            "doc_id": str(doc_id),
            "token_start": token_start,
            "char_start": char_start,
            "char_end": char_end,
            "input_ids": np.asarray(local_ids, dtype=np.int64),
            "char_ids": char_ids.astype(np.int64, copy=False),
            "char_to_token": local_char_to_token.astype(
                np.int64, copy=False
            ),
            "start_targets": start_targets,
            "end_targets": end_targets,
        })

    if include_labels:
        missing = set(spans) - covered_spans
    else:
        missing = set()

    return windows, covered_spans, missing


# ---------- Build training and validation windows ----------
S1_TRAIN_WINDOWS = []
S1_VAL_WINDOWS = []
S1_TRAIN_COVERAGE_ERRORS = {}
S1_VAL_COVERAGE_ERRORS = {}

for doc_id in S1_TRAIN_DOC_IDS:
    windows, covered, missing = s1_build_doc_windows(
        doc_id, include_labels=True
    )
    S1_TRAIN_WINDOWS.extend(windows)
    if missing:
        S1_TRAIN_COVERAGE_ERRORS[doc_id] = sorted(missing)

for doc_id in S1_VAL_DOC_IDS:
    windows, covered, missing = s1_build_doc_windows(
        doc_id, include_labels=True
    )
    S1_VAL_WINDOWS.extend(windows)
    if missing:
        S1_VAL_COVERAGE_ERRORS[doc_id] = sorted(missing)

S1_TRAIN_WINDOW_LENGTHS = [
    (len(w["input_ids"]), len(w["char_ids"]))
    for w in S1_TRAIN_WINDOWS
]
S1_VAL_WINDOW_LENGTHS = [
    (len(w["input_ids"]), len(w["char_ids"]))
    for w in S1_VAL_WINDOWS
]

S1_TRAIN_SPANS = set(
    span
    for doc_id in S1_TRAIN_DOC_IDS
    for span in S1_SPANS_BY_DOC[doc_id]
)
S1_VAL_SPANS = set(
    span
    for doc_id in S1_VAL_DOC_IDS
    for span in S1_SPANS_BY_DOC[doc_id]
)

print("\n" + "=" * 68)
print("S1 REDO — WINDOW COVERAGE REPORT")
print("=" * 68)
print("Training windows:", len(S1_TRAIN_WINDOWS))
print("Validation windows:", len(S1_VAL_WINDOWS))
print(
    "Train token length min/max:",
    min(x[0] for x in S1_TRAIN_WINDOW_LENGTHS),
    max(x[0] for x in S1_TRAIN_WINDOW_LENGTHS),
)
print(
    "Train character length median/max:",
    int(np.median([x[1] for x in S1_TRAIN_WINDOW_LENGTHS])),
    max(x[1] for x in S1_TRAIN_WINDOW_LENGTHS),
)
print(
    "Validation character length median/max:",
    int(np.median([x[1] for x in S1_VAL_WINDOW_LENGTHS])),
    max(x[1] for x in S1_VAL_WINDOW_LENGTHS),
)
print("Training gold spans:", len(S1_TRAIN_SPANS))
print("Training spans not covered by any window:", sum(
    len(x) for x in S1_TRAIN_COVERAGE_ERRORS.values()
))
print("Validation gold spans:", len(S1_VAL_SPANS))
print("Validation spans not covered by any window:", sum(
    len(x) for x in S1_VAL_COVERAGE_ERRORS.values()
))

if S1_TRAIN_COVERAGE_ERRORS or S1_VAL_COVERAGE_ERRORS:
    print("Training coverage examples:",
          list(S1_TRAIN_COVERAGE_ERRORS.items())[:3])
    print("Validation coverage examples:",
          list(S1_VAL_COVERAGE_ERRORS.items())[:3])
    raise RuntimeError(
        "Some annotated spans do not fit fully within any token window. "
        "We should fix windowing before training."
    )

print("All gold spans are covered by at least one window.")
print("Cell 7 finished. No fine-tuning performed.")

S1 REDO — WINDOW AND CHARACTER-TARGET SETUP
Window tokens: 512
Window stride: 384
Character vocabulary size: 159

S1 REDO — WINDOW COVERAGE REPORT
Training windows: 903
Validation windows: 293
Train token length min/max: 512 512
Train character length median/max: 2530 2886
Validation character length median/max: 2494 2956
Training gold spans: 6499
Training spans not covered by any window: 0
Validation gold spans: 1858
Validation spans not covered by any window: 0
All gold spans are covered by at least one window.
Cell 7 finished. No fine-tuning performed.


In [8]:
# ============================================================
# S1 REDO — CELL 8
# Purpose:
#   1. Create a character-level NER head on Architecture A.
#   2. Build padded batches and training/validation DataLoaders.
#   3. Calculate class weights for label imbalance.
#   4. Run a forward/backward smoke test before full training.
# ============================================================

import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

assert "S1_A_MODEL" in globals()
assert "S1_TRAIN_WINDOWS" in globals()
assert "S1_VAL_WINDOWS" in globals()
assert "S1_CHAR_VOCAB_SIZE" in globals()

S1_CHAR_EMBED_DIM = 32
S1_HEAD_HIDDEN_DIM = 256
S1_HEAD_DROPOUT = 0.15
S1_BATCH_SIZE = 1
S1_NUM_WORKERS = 0


# ---------- Dataset wrapper ----------
class S1NERDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        return self.windows[index]


# ---------- Batch collation ----------
def s1_collate_windows(items):
    batch_size = len(items)
    max_tokens = max(len(x["input_ids"]) for x in items)
    max_chars = max(len(x["char_ids"]) for x in items)

    input_ids = torch.zeros(
        (batch_size, max_tokens), dtype=torch.long
    )
    char_ids = torch.zeros(
        (batch_size, max_chars), dtype=torch.long
    )
    char_to_token = torch.zeros(
        (batch_size, max_chars), dtype=torch.long
    )
    start_targets = torch.zeros(
        (batch_size, max_chars, S1_NUM_LABELS), dtype=torch.float32
    )
    end_targets = torch.zeros_like(start_targets)
    char_mask = torch.zeros(
        (batch_size, max_chars), dtype=torch.float32
    )

    doc_ids, char_starts, char_ends = [], [], []

    for i, item in enumerate(items):
        nt = len(item["input_ids"])
        nc = len(item["char_ids"])

        input_ids[i, :nt] = torch.as_tensor(
            item["input_ids"], dtype=torch.long
        )
        char_ids[i, :nc] = torch.as_tensor(
            item["char_ids"], dtype=torch.long
        )
        char_to_token[i, :nc] = torch.as_tensor(
            item["char_to_token"], dtype=torch.long
        )
        start_targets[i, :nc] = torch.as_tensor(
            item["start_targets"], dtype=torch.float32
        )
        end_targets[i, :nc] = torch.as_tensor(
            item["end_targets"], dtype=torch.float32
        )
        char_mask[i, :nc] = 1.0

        doc_ids.append(str(item["doc_id"]))
        char_starts.append(int(item["char_start"]))
        char_ends.append(int(item["char_end"]))

    return {
        "input_ids": input_ids,
        "char_ids": char_ids,
        "char_to_token": char_to_token,
        "start_targets": start_targets,
        "end_targets": end_targets,
        "char_mask": char_mask,
        "doc_ids": doc_ids,
        "char_starts": char_starts,
        "char_ends": char_ends,
    }


def s1_to_device(batch):
    return {
        key: value.to(S1_DEVICE, non_blocking=True)
        if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


S1_TRAIN_DATASET = S1NERDataset(S1_TRAIN_WINDOWS)
S1_VAL_DATASET = S1NERDataset(S1_VAL_WINDOWS)

S1_TRAIN_LOADER = DataLoader(
    S1_TRAIN_DATASET,
    batch_size=S1_BATCH_SIZE,
    shuffle=True,
    num_workers=S1_NUM_WORKERS,
    collate_fn=s1_collate_windows,
    pin_memory=(S1_DEVICE.type == "cuda"),
)

S1_VAL_LOADER = DataLoader(
    S1_VAL_DATASET,
    batch_size=1,
    shuffle=False,
    num_workers=S1_NUM_WORKERS,
    collate_fn=s1_collate_windows,
    pin_memory=(S1_DEVICE.type == "cuda"),
)


# ---------- Character-level NER head ----------
class S1CharacterNER(nn.Module):
    def __init__(
        self,
        backbone,
        char_vocab_size,
        char_embed_dim=32,
        hidden_dim=256,
        dropout=0.15,
    ):
        super().__init__()

        self.backbone = backbone
        backbone_dim = int(S1_A_CONFIG["d_model"])

        self.char_embedding = nn.Embedding(
            char_vocab_size,
            char_embed_dim,
            padding_idx=S1_CHAR_PAD_ID,
        )

        self.fusion = nn.Linear(
            backbone_dim + char_embed_dim, hidden_dim
        )
        self.fusion_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Two residual character-convolution blocks refine local boundaries.
        self.conv1 = nn.Conv1d(
            hidden_dim, hidden_dim, kernel_size=3, padding=1
        )
        self.conv1_norm = nn.LayerNorm(hidden_dim)

        self.conv2 = nn.Conv1d(
            hidden_dim, hidden_dim, kernel_size=3, padding=1
        )
        self.conv2_norm = nn.LayerNorm(hidden_dim)

        self.start_head = nn.Linear(hidden_dim, S1_NUM_LABELS)
        self.end_head = nn.Linear(hidden_dim, S1_NUM_LABELS)

    def forward(self, input_ids, char_ids, char_to_token):
        # Backbone representation for each input token: [B, T, D].
        token_hidden = self.backbone(input_ids)
        batch_size, char_len = char_ids.shape

        # Gather one contextual token representation for every character.
        gather_index = char_to_token.unsqueeze(-1).expand(
            batch_size, char_len, token_hidden.shape[-1]
        )
        char_context = token_hidden.gather(1, gather_index)

        char_features = self.char_embedding(char_ids)
        x = torch.cat([char_context, char_features], dim=-1)

        x = self.dropout(F.gelu(self.fusion_norm(self.fusion(x))))

        residual = x
        x = self.conv1(x.transpose(1, 2)).transpose(1, 2)
        x = self.conv1_norm(F.gelu(x) + residual)
        x = self.dropout(x)

        residual = x
        x = self.conv2(x.transpose(1, 2)).transpose(1, 2)
        x = self.conv2_norm(F.gelu(x) + residual)
        x = self.dropout(x)

        start_logits = self.start_head(x)
        end_logits = self.end_head(x)

        return start_logits, end_logits


# ---------- Instantiate model with the full pretrained backbone ----------
S1_NER_MODEL = S1CharacterNER(
    backbone=S1_A_MODEL,
    char_vocab_size=S1_CHAR_VOCAB_SIZE,
    char_embed_dim=S1_CHAR_EMBED_DIM,
    hidden_dim=S1_HEAD_HIDDEN_DIM,
    dropout=S1_HEAD_DROPOUT,
).to(S1_DEVICE)

# Full fine-tuning: do not freeze any Transformer layer.
for parameter in S1_NER_MODEL.backbone.parameters():
    parameter.requires_grad = True

# ---------- Class-balancing weights ----------
S1_START_POS_COUNTS = np.zeros(S1_NUM_LABELS, dtype=np.float64)
S1_END_POS_COUNTS = np.zeros(S1_NUM_LABELS, dtype=np.float64)
S1_TOTAL_CHAR_POSITIONS = 0

for window in S1_TRAIN_WINDOWS:
    S1_START_POS_COUNTS += window["start_targets"].sum(axis=0)
    S1_END_POS_COUNTS += window["end_targets"].sum(axis=0)
    S1_TOTAL_CHAR_POSITIONS += len(window["char_ids"])

S1_POS_COUNTS = (S1_START_POS_COUNTS + S1_END_POS_COUNTS) / 2.0
S1_NEG_COUNTS = np.maximum(
    S1_TOTAL_CHAR_POSITIONS - S1_POS_COUNTS, 1.0
)

# Fourth-root weighting reduces imbalance without the extreme weights of
# neg/pos ratios; values are bounded to control false positives.
S1_POS_WEIGHTS = np.clip(
    np.power(
        S1_NEG_COUNTS / np.maximum(S1_POS_COUNTS, 1.0),
        0.25,
    ),
    2.0,
    10.0,
).astype(np.float32)

S1_POS_WEIGHTS_TENSOR = torch.tensor(
    S1_POS_WEIGHTS, dtype=torch.float32, device=S1_DEVICE
)


# ---------- Smoke test: forward, finite loss, backward ----------
print("=" * 68)
print("S1 REDO — NER HEAD AND MEMORY SMOKE TEST")
print("=" * 68)

S1_SMOKE_BATCH = next(iter(S1_TRAIN_LOADER))
S1_SMOKE_BATCH_DEVICE = s1_to_device(S1_SMOKE_BATCH)

S1_NER_MODEL.train()
S1_NER_MODEL.zero_grad(set_to_none=True)

try:
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(S1_DEVICE.type == "cuda"),
    ):
        S1_SMOKE_START, S1_SMOKE_END = S1_NER_MODEL(
            S1_SMOKE_BATCH_DEVICE["input_ids"],
            S1_SMOKE_BATCH_DEVICE["char_ids"],
            S1_SMOKE_BATCH_DEVICE["char_to_token"],
        )

        S1_SMOKE_MASK = S1_SMOKE_BATCH_DEVICE["char_mask"].unsqueeze(-1)
        S1_SMOKE_START_TARGET = S1_SMOKE_BATCH_DEVICE["start_targets"]
        S1_SMOKE_END_TARGET = S1_SMOKE_BATCH_DEVICE["end_targets"]

        S1_SMOKE_START_LOSS = F.binary_cross_entropy_with_logits(
            S1_SMOKE_START.float(),
            S1_SMOKE_START_TARGET,
            pos_weight=S1_POS_WEIGHTS_TENSOR,
            reduction="none",
        )
        S1_SMOKE_END_LOSS = F.binary_cross_entropy_with_logits(
            S1_SMOKE_END.float(),
            S1_SMOKE_END_TARGET,
            pos_weight=S1_POS_WEIGHTS_TENSOR,
            reduction="none",
        )

        S1_SMOKE_DENOM = (
            S1_SMOKE_BATCH_DEVICE["char_mask"].sum()
            * S1_NUM_LABELS * 2
        ).clamp_min(1.0)

        S1_SMOKE_LOSS = (
            (S1_SMOKE_START_LOSS + S1_SMOKE_END_LOSS)
            * S1_SMOKE_MASK
        ).sum() / S1_SMOKE_DENOM

    if not torch.isfinite(S1_SMOKE_LOSS):
        raise RuntimeError("Smoke-test loss is not finite.")

    S1_SMOKE_LOSS.backward()

    S1_BACKBONE_GRAD_EXISTS = any(
        parameter.grad is not None
        and torch.isfinite(parameter.grad).all().item()
        for parameter in S1_NER_MODEL.backbone.parameters()
        if parameter.requires_grad
    )
    S1_HEAD_GRAD_EXISTS = any(
        parameter.grad is not None
        and torch.isfinite(parameter.grad).all().item()
        for name, parameter in S1_NER_MODEL.named_parameters()
        if not name.startswith("backbone.")
    )

    if not S1_BACKBONE_GRAD_EXISTS or not S1_HEAD_GRAD_EXISTS:
        raise RuntimeError(
            "Gradient smoke test failed for the backbone or NER head."
        )

    print("Input token shape:", tuple(S1_SMOKE_BATCH_DEVICE["input_ids"].shape))
    print("Character shape:", tuple(S1_SMOKE_BATCH_DEVICE["char_ids"].shape))
    print("Start logits:", tuple(S1_SMOKE_START.shape))
    print("End logits:", tuple(S1_SMOKE_END.shape))
    print(f"Smoke-test loss: {S1_SMOKE_LOSS.item():.6f}")
    print("Backbone gradients: finite and present")
    print("NER-head gradients: finite and present")

finally:
    S1_NER_MODEL.zero_grad(set_to_none=True)
    del S1_SMOKE_BATCH_DEVICE, S1_SMOKE_START, S1_SMOKE_END
    del S1_SMOKE_START_LOSS, S1_SMOKE_END_LOSS
    del S1_SMOKE_LOSS
    gc.collect()
    if S1_DEVICE.type == "cuda":
        torch.cuda.empty_cache()

S1_NER_MODEL.eval()

print(f"\nTraining windows: {len(S1_TRAIN_DATASET)}")
print(f"Validation windows: {len(S1_VAL_DATASET)}")
print("All 12 labels have class weights:", len(S1_POS_WEIGHTS_TENSOR) == 12)
print("Full fine-tuning enabled:", all(
    parameter.requires_grad
    for parameter in S1_NER_MODEL.backbone.parameters()
))
print("Cell 8 finished. No optimizer steps taken.")

S1 REDO — NER HEAD AND MEMORY SMOKE TEST
Input token shape: (1, 512)
Character shape: (1, 2485)
Start logits: (1, 2485, 12)
End logits: (1, 2485, 12)
Smoke-test loss: 0.740499
Backbone gradients: finite and present
NER-head gradients: finite and present

Training windows: 903
Validation windows: 293
All 12 labels have class weights: True
Full fine-tuning enabled: True
Cell 8 finished. No optimizer steps taken.


In [9]:
# ============================================================
# S1 REDO — CELL 9
# Purpose:
#   1. Fully fine-tune all Architecture A parameters.
#   2. Use mixed precision, gradient accumulation, and clipping.
#   3. Validate after every epoch.
#   4. Save a model checkpoint for every epoch for later F1 selection.
# ============================================================

import os
import json
import gc
import numpy as np
import torch
import torch.nn.functional as F

assert "S1_NER_MODEL" in globals()
assert "S1_TRAIN_LOADER" in globals()
assert "S1_VAL_LOADER" in globals()

S1_RUN_DIR = Path("/kaggle/working/S1_A_FullFineTuning")
S1_CHECKPOINT_DIR = S1_RUN_DIR / "checkpoints"
S1_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

S1_EPOCHS = 3
S1_GRAD_ACCUM_STEPS = 3
S1_BACKBONE_LR = 1e-5
S1_HEAD_LR = 2e-4
S1_WEIGHT_DECAY = 0.01
S1_MAX_GRAD_NORM = 1.0

# Explicitly confirm every backbone parameter is trainable.
for p in S1_NER_MODEL.backbone.parameters():
    p.requires_grad = True

S1_BACKBONE_PARAMS = [
    p for name, p in S1_NER_MODEL.named_parameters()
    if name.startswith("backbone.") and p.requires_grad
]
S1_HEAD_PARAMS = [
    p for name, p in S1_NER_MODEL.named_parameters()
    if not name.startswith("backbone.") and p.requires_grad
]

S1_OPTIMIZER = torch.optim.AdamW(
    [
        {"params": S1_BACKBONE_PARAMS, "lr": S1_BACKBONE_LR},
        {"params": S1_HEAD_PARAMS, "lr": S1_HEAD_LR},
    ],
    weight_decay=S1_WEIGHT_DECAY,
    fused=(S1_DEVICE.type == "cuda"),
)

S1_SCALER = torch.amp.GradScaler(
    "cuda", enabled=(S1_DEVICE.type == "cuda")
)

def s1_loss_for_batch(model, batch_device):
    start_logits, end_logits = model(
        batch_device["input_ids"],
        batch_device["char_ids"],
        batch_device["char_to_token"],
    )

    start_logits = start_logits.float()
    end_logits = end_logits.float()
    start_targets = batch_device["start_targets"].float()
    end_targets = batch_device["end_targets"].float()

    mask = batch_device["char_mask"].unsqueeze(-1)
    pos_weights = S1_POS_WEIGHTS_TENSOR.float()

    start_bce = F.binary_cross_entropy_with_logits(
        start_logits,
        start_targets,
        pos_weight=pos_weights,
        reduction="none",
    )
    end_bce = F.binary_cross_entropy_with_logits(
        end_logits,
        end_targets,
        pos_weight=pos_weights,
        reduction="none",
    )

    denom = (
        batch_device["char_mask"].sum()
        * S1_NUM_LABELS
        * 2
    ).clamp_min(1.0)

    loss = ((start_bce + end_bce) * mask).sum() / denom
    return loss


@torch.no_grad()
def s1_validate_loss():
    S1_NER_MODEL.eval()
    total_loss = 0.0
    count = 0

    for batch in S1_VAL_LOADER:
        batch_device = s1_to_device(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S1_DEVICE.type == "cuda"),
        ):
            loss = s1_loss_for_batch(S1_NER_MODEL, batch_device)

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite validation loss detected.")

        total_loss += float(loss.item())
        count += 1

    return total_loss / max(count, 1)


S1_HISTORY = []
S1_BEST_VAL_LOSS = float("inf")
S1_BEST_LOSS_EPOCH = None

print("=" * 68)
print("S1 REDO — ARCHITECTURE A FULL FINE-TUNING")
print("=" * 68)
print(f"Training windows: {len(S1_TRAIN_LOADER)}")
print(f"Validation windows: {len(S1_VAL_LOADER)}")
print(f"Epochs: {S1_EPOCHS}")
print(f"Backbone LR: {S1_BACKBONE_LR}")
print(f"NER-head LR: {S1_HEAD_LR}")
print(f"Gradient accumulation: {S1_GRAD_ACCUM_STEPS}")
print("Trainable backbone parameters:", f"{sum(p.numel() for p in S1_BACKBONE_PARAMS):,}")
print("Trainable NER-head parameters:", f"{sum(p.numel() for p in S1_HEAD_PARAMS):,}")
print("-" * 68)

for epoch in range(1, S1_EPOCHS + 1):
    S1_NER_MODEL.train()
    S1_OPTIMIZER.zero_grad(set_to_none=True)

    running_loss = 0.0
    optimizer_steps = 0

    for batch_idx, batch in enumerate(S1_TRAIN_LOADER, start=1):
        batch_device = s1_to_device(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S1_DEVICE.type == "cuda"),
        ):
            loss = s1_loss_for_batch(S1_NER_MODEL, batch_device)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}, "
                f"batch {batch_idx}: {loss.item()}"
            )

        S1_SCALER.scale(
            loss / S1_GRAD_ACCUM_STEPS
        ).backward()

        running_loss += float(loss.detach().item())

        if (
            batch_idx % S1_GRAD_ACCUM_STEPS == 0
            or batch_idx == len(S1_TRAIN_LOADER)
        ):
            S1_SCALER.unscale_(S1_OPTIMIZER)

            torch.nn.utils.clip_grad_norm_(
                S1_NER_MODEL.parameters(),
                max_norm=S1_MAX_GRAD_NORM,
            )

            S1_SCALER.step(S1_OPTIMIZER)
            S1_SCALER.update()
            S1_OPTIMIZER.zero_grad(set_to_none=True)
            optimizer_steps += 1

        if batch_idx % 150 == 0 or batch_idx == len(S1_TRAIN_LOADER):
            print(
                f"Epoch {epoch}/{S1_EPOCHS} | "
                f"batch {batch_idx}/{len(S1_TRAIN_LOADER)} | "
                f"train loss={running_loss / batch_idx:.5f}"
            )

    S1_VAL_LOSS = s1_validate_loss()
    S1_EPOCH_TRAIN_LOSS = running_loss / len(S1_TRAIN_LOADER)

    S1_RECORD = {
        "epoch": epoch,
        "train_loss": S1_EPOCH_TRAIN_LOSS,
        "validation_loss": S1_VAL_LOSS,
        "optimizer_steps": optimizer_steps,
        "backbone_lr": S1_BACKBONE_LR,
        "head_lr": S1_HEAD_LR,
        "gradient_accumulation": S1_GRAD_ACCUM_STEPS,
    }
    S1_HISTORY.append(S1_RECORD)

    # Keep every epoch for later strict-span-F1 comparison.
    # Saving the state dict directly preserves tied-storage serialization.
    S1_EPOCH_CKPT = S1_CHECKPOINT_DIR / f"epoch_{epoch}.pt"
    torch.save(
        {
            "model_state_dict": S1_NER_MODEL.state_dict(),
            "epoch": epoch,
            "train_loss": S1_EPOCH_TRAIN_LOSS,
            "validation_loss": S1_VAL_LOSS,
            "labels": S1_LABEL_TO_ID,
            "character_vocab": S1_CHAR_TO_ID,
            "architecture_config": S1_A_CONFIG,
            "window_tokens": S1_WINDOW_TOKENS,
            "window_stride": S1_WINDOW_STRIDE,
        },
        S1_EPOCH_CKPT,
    )

    print(
        f"Epoch {epoch} complete | "
        f"train loss={S1_EPOCH_TRAIN_LOSS:.5f} | "
        f"validation loss={S1_VAL_LOSS:.5f} | "
        f"optimizer steps={optimizer_steps}"
    )
    print(f"Checkpoint saved: {S1_EPOCH_CKPT}")

    if S1_VAL_LOSS < S1_BEST_VAL_LOSS:
        S1_BEST_VAL_LOSS = S1_VAL_LOSS
        S1_BEST_LOSS_EPOCH = epoch

    with open(S1_RUN_DIR / "training_history.json", "w", encoding="utf-8") as f:
        json.dump(S1_HISTORY, f, indent=2)

    gc.collect()
    if S1_DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\n" + "=" * 68)
print("S1 FULL FINE-TUNING COMPLETE")
print("=" * 68)
print("Best epoch by validation loss:", S1_BEST_LOSS_EPOCH)
print(f"Best validation loss: {S1_BEST_VAL_LOSS:.5f}")
print("Checkpoint directory:", S1_CHECKPOINT_DIR)
print("Checkpoints saved:", sorted(p.name for p in S1_CHECKPOINT_DIR.glob("epoch_*.pt")))
print("Training history:", S1_RUN_DIR / "training_history.json")
print("\nNext step: evaluate all saved epochs with exact-span Macro-F1.")

S1 REDO — ARCHITECTURE A FULL FINE-TUNING
Training windows: 903
Validation windows: 293
Epochs: 3
Backbone LR: 1e-05
NER-head LR: 0.0002
Gradient accumulation: 3
Trainable backbone parameters: 125,028,864
Trainable NER-head parameters: 611,576
--------------------------------------------------------------------
Epoch 1/3 | batch 150/903 | train loss=0.17729
Epoch 1/3 | batch 300/903 | train loss=0.10570
Epoch 1/3 | batch 450/903 | train loss=0.07818
Epoch 1/3 | batch 600/903 | train loss=0.06363
Epoch 1/3 | batch 750/903 | train loss=0.05451
Epoch 1/3 | batch 900/903 | train loss=0.04812
Epoch 1/3 | batch 903/903 | train loss=0.04800
Epoch 1 complete | train loss=0.04800 | validation loss=0.01522 | optimizer steps=301
Checkpoint saved: /kaggle/working/S1_A_FullFineTuning/checkpoints/epoch_1.pt
Epoch 2/3 | batch 150/903 | train loss=0.01545
Epoch 2/3 | batch 300/903 | train loss=0.01580
Epoch 2/3 | batch 450/903 | train loss=0.01630
Epoch 2/3 | batch 600/903 | train loss=0.01617
Epoch 2

In [10]:
# ============================================================
# S1 REDO — CELL 10
# Purpose:
#   1. Evaluate each saved epoch on the held-out validation set.
#   2. Aggregate overlapping-window predictions in character space.
#   3. Measure exact-span Macro-F1 and Micro-F1.
#   4. Tune global and per-label boundary thresholds.
#   5. Select the epoch with the best validation Macro-F1.
# ============================================================

import json
import gc
import numpy as np
import torch
from pathlib import Path

assert "S1_NER_MODEL" in globals()
assert "S1_VAL_LOADER" in globals()
assert "S1_VAL_DOC_IDS" in globals()
assert "S1_SPANS_BY_DOC" in globals()

S1_CHECKPOINT_DIR = Path("/kaggle/working/S1_A_FullFineTuning/checkpoints")
S1_EVAL_CHECKPOINTS = [
    S1_CHECKPOINT_DIR / f"epoch_{epoch}.pt"
    for epoch in (1, 2, 3)
]
for checkpoint_path in S1_EVAL_CHECKPOINTS:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing epoch checkpoint: {checkpoint_path}")

# IMPORTANT: include document ID in each gold key. Same offsets in different
# documents are distinct entities and must never be deduplicated together.
S1_VAL_GOLD_EXACT = {
    (str(doc_id), int(start), int(end), str(label))
    for doc_id in S1_VAL_DOC_IDS
    for start, end, label in S1_SPANS_BY_DOC[doc_id]
}
S1_VAL_GOLD_BY_LABEL = {
    label: {span for span in S1_VAL_GOLD_EXACT if span[3] == label}
    for label in S1_LABELS
}

S1_THRESHOLD_GRID = np.round(np.arange(0.10, 0.901, 0.05), 2)


# ---------- Aggregate prediction logits for one checkpoint ----------
@torch.no_grad()
def s1_infer_validation_aggregate():
    S1_NER_MODEL.eval()

    aggregate = {
        str(doc_id): {
            "start_sum": np.zeros(
                (len(S1_ALL_TEXTS[str(doc_id)]), S1_NUM_LABELS),
                dtype=np.float32
            ),
            "end_sum": np.zeros(
                (len(S1_ALL_TEXTS[str(doc_id)]), S1_NUM_LABELS),
                dtype=np.float32
            ),
            "count": np.zeros(
                len(S1_ALL_TEXTS[str(doc_id)]), dtype=np.float32
            ),
        }
        for doc_id in S1_VAL_DOC_IDS
    }

    for batch_idx, batch in enumerate(S1_VAL_LOADER, start=1):
        batch_device = s1_to_device(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S1_DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = S1_NER_MODEL(
                batch_device["input_ids"],
                batch_device["char_ids"],
                batch_device["char_to_token"],
            )

        start_logits = start_logits.float().cpu().numpy()
        end_logits = end_logits.float().cpu().numpy()

        for i, raw_doc_id in enumerate(batch["doc_ids"]):
            doc_id = str(raw_doc_id)
            char_start = int(batch["char_starts"][i])
            char_end = int(batch["char_ends"][i])
            doc_len = len(S1_ALL_TEXTS[doc_id])

            n = min(
                char_end - char_start,
                start_logits.shape[1],
                end_logits.shape[1],
                max(0, doc_len - char_start)
            )
            if n <= 0:
                continue

            d = aggregate[doc_id]
            d["start_sum"][char_start:char_start+n] += start_logits[i, :n]
            d["end_sum"][char_start:char_start+n] += end_logits[i, :n]
            d["count"][char_start:char_start+n] += 1.0

        if batch_idx % 100 == 0 or batch_idx == len(S1_VAL_LOADER):
            print(f"  Validation inference: {batch_idx}/{len(S1_VAL_LOADER)}")

    # Average logits before applying sigmoid.
    for doc_id, d in aggregate.items():
        denom = np.maximum(d["count"][:, None], 1.0)
        avg_start = d.pop("start_sum") / denom
        avg_end = d.pop("end_sum") / denom
        d["start"] = 1.0 / (
            1.0 + np.exp(-np.clip(avg_start, -30, 30))
        )
        d["end"] = 1.0 / (
            1.0 + np.exp(-np.clip(avg_end, -30, 30))
        )

    return aggregate


# ---------- Decode start/end boundaries into character spans ----------
def s1_decode_label(aggregate, label, threshold):
    label_id = S1_LABEL_TO_ID[label]
    predicted = set()

    for doc_id, d in aggregate.items():
        start_positions = np.flatnonzero(
            d["start"][:, label_id] >= threshold
        )
        end_positions = np.flatnonzero(
            d["end"][:, label_id] >= threshold
        )

        # Pair each start with the first unused end at or after it.
        end_idx = 0
        for start in start_positions:
            while (
                end_idx < len(end_positions)
                and end_positions[end_idx] < start
            ):
                end_idx += 1

            if end_idx >= len(end_positions):
                break

            end_exclusive = int(end_positions[end_idx]) + 1
            end_idx += 1

            if 0 <= int(start) < end_exclusive <= len(S1_ALL_TEXTS[doc_id]):
                predicted.add((
                    str(doc_id), int(start), end_exclusive, label
                ))

    return predicted


def s1_decode_all_labels(aggregate, threshold_or_dict):
    predicted = set()
    for label in S1_LABELS:
        threshold = (
            float(threshold_or_dict[label])
            if isinstance(threshold_or_dict, dict)
            else float(threshold_or_dict)
        )
        predicted.update(s1_decode_label(aggregate, label, threshold))
    return predicted


def s1_exact_span_scores(predicted):
    per_label = {}
    all_tp = 0

    for label in S1_LABELS:
        gold = S1_VAL_GOLD_BY_LABEL[label]
        pred = {span for span in predicted if span[3] == label}
        tp = len(gold & pred)
        all_tp += tp

        precision = tp / len(pred) if pred else 0.0
        recall = tp / len(gold) if gold else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        per_label[label] = {
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "gold": len(gold),
            "predicted": len(pred),
            "true_positive": tp,
        }

    macro_f1 = float(np.mean([per_label[label]["f1"] for label in S1_LABELS]))
    micro_precision = all_tp / len(predicted) if predicted else 0.0
    micro_recall = all_tp / len(S1_VAL_GOLD_EXACT) if S1_VAL_GOLD_EXACT else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if micro_precision + micro_recall else 0.0
    )

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "gold_count": len(S1_VAL_GOLD_EXACT),
        "predicted_count": len(predicted),
        "per_label": per_label,
    }


# ---------- Calibrate a global and one threshold per label ----------
def s1_calibrate_thresholds(aggregate):
    global_results = []

    for threshold in S1_THRESHOLD_GRID:
        predicted = s1_decode_all_labels(aggregate, float(threshold))
        scores = s1_exact_span_scores(predicted)
        global_results.append({
            "threshold": float(threshold),
            "scores": scores,
        })

    # Deterministic tie-break: prefer the higher threshold.
    global_results.sort(
        key=lambda item: (
            item["scores"]["macro_f1"], item["threshold"]
        ),
        reverse=True,
    )
    global_best = global_results[0]

    label_thresholds = {}
    label_predictions = set()
    label_rows = []

    for label in S1_LABELS:
        gold = S1_VAL_GOLD_BY_LABEL[label]
        best = None

        for threshold in S1_THRESHOLD_GRID:
            pred_label = s1_decode_label(
                aggregate, label, float(threshold)
            )
            tp = len(gold & pred_label)
            precision = tp / len(pred_label) if pred_label else 0.0
            recall = tp / len(gold) if gold else 0.0
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall else 0.0
            )

            candidate = {
                "threshold": float(threshold),
                "f1": f1,
                "precision": precision,
                "recall": recall,
                "predicted": len(pred_label),
                "predictions": pred_label,
            }

            if best is None or (
                candidate["f1"], candidate["threshold"]
            ) > (best["f1"], best["threshold"]):
                best = candidate

        label_thresholds[label] = best["threshold"]
        label_predictions.update(best["predictions"])
        label_rows.append({
            "label": label,
            "threshold": best["threshold"],
            "f1": best["f1"],
            "precision": best["precision"],
            "recall": best["recall"],
            "predicted": best["predicted"],
        })

    tuned_scores = s1_exact_span_scores(label_predictions)

    return {
        "global_best": global_best,
        "global_results": global_results,
        "label_thresholds": label_thresholds,
        "label_scores": tuned_scores,
        "label_rows": label_rows,
    }


# ---------- Evaluate every saved epoch checkpoint ----------
S1_EPOCH_EVALUATIONS = []

print("=" * 68)
print("S1 REDO — EXACT-SPAN VALIDATION")
print("=" * 68)
print("Validation documents:", len(S1_VAL_DOC_IDS))
print("Exact gold spans:", len(S1_VAL_GOLD_EXACT))
print("Labels:", len(S1_LABELS))

for checkpoint_path in S1_EVAL_CHECKPOINTS:
    epoch_checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    if epoch_checkpoint.get("labels") != S1_LABEL_TO_ID:
        raise ValueError(f"Label mapping mismatch in {checkpoint_path.name}")

    S1_NER_MODEL.load_state_dict(
        epoch_checkpoint["model_state_dict"],
        strict=True,
    )
    del epoch_checkpoint
    S1_NER_MODEL.eval()

    print(f"\nEvaluating {checkpoint_path.name}...")
    S1_EPOCH_AGGREGATE = s1_infer_validation_aggregate()
    S1_EPOCH_CALIBRATION = s1_calibrate_thresholds(S1_EPOCH_AGGREGATE)

    S1_GLOBAL_BEST = S1_EPOCH_CALIBRATION["global_best"]
    S1_TUNED_SCORES = S1_EPOCH_CALIBRATION["label_scores"]

    S1_EPOCH_EVALUATIONS.append({
        "epoch": int(checkpoint_path.stem.split("_")[-1]),
        "checkpoint_path": str(checkpoint_path),
        "global_best_threshold": S1_GLOBAL_BEST["threshold"],
        "global_macro_f1": S1_GLOBAL_BEST["scores"]["macro_f1"],
        "global_micro_f1": S1_GLOBAL_BEST["scores"]["micro_f1"],
        "global_precision": S1_GLOBAL_BEST["scores"]["micro_precision"],
        "global_recall": S1_GLOBAL_BEST["scores"]["micro_recall"],
        "label_tuned_macro_f1": S1_TUNED_SCORES["macro_f1"],
        "label_tuned_micro_f1": S1_TUNED_SCORES["micro_f1"],
        "label_tuned_precision": S1_TUNED_SCORES["micro_precision"],
        "label_tuned_recall": S1_TUNED_SCORES["micro_recall"],
        "label_thresholds": S1_EPOCH_CALIBRATION["label_thresholds"],
        "label_rows": S1_EPOCH_CALIBRATION["label_rows"],
    })

    print(
        f"  Global threshold={S1_GLOBAL_BEST['threshold']:.2f} | "
        f"Macro-F1={S1_GLOBAL_BEST['scores']['macro_f1']:.4f} | "
        f"Micro-F1={S1_GLOBAL_BEST['scores']['micro_f1']:.4f}"
    )
    print(
        f"  Label-tuned Macro-F1={S1_TUNED_SCORES['macro_f1']:.4f} | "
        f"Micro-F1={S1_TUNED_SCORES['micro_f1']:.4f}"
    )

    del S1_EPOCH_AGGREGATE, S1_EPOCH_CALIBRATION
    gc.collect()
    if S1_DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# Select checkpoint using validation label-tuned Macro-F1.
S1_EPOCH_EVALUATIONS.sort(
    key=lambda x: (x["label_tuned_macro_f1"], x["epoch"]),
    reverse=True,
)
S1_SELECTED_EVALUATION = S1_EPOCH_EVALUATIONS[0]
S1_SELECTED_EPOCH = S1_SELECTED_EVALUATION["epoch"]
S1_SELECTED_CHECKPOINT = Path(S1_SELECTED_EVALUATION["checkpoint_path"])
S1_SELECTED_LABEL_THRESHOLDS = S1_SELECTED_EVALUATION["label_thresholds"]

S1_EVALUATION_SUMMARY_PATH = Path(
    "/kaggle/working/S1_A_FullFineTuning/strict_span_validation.json"
)
with open(S1_EVALUATION_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(S1_EPOCH_EVALUATIONS, f, indent=2)

print("\n" + "=" * 68)
print("S1 CHECKPOINT COMPARISON")
print("=" * 68)
print(
    f"{'Epoch':<8} {'Global Macro-F1':<18} "
    f"{'Tuned Macro-F1':<18} {'Tuned Micro-F1':<18}"
)
for result in sorted(S1_EPOCH_EVALUATIONS, key=lambda x: x["epoch"]):
    print(
        f"{result['epoch']:<8} "
        f"{result['global_macro_f1']:<18.4f} "
        f"{result['label_tuned_macro_f1']:<18.4f} "
        f"{result['label_tuned_micro_f1']:<18.4f}"
    )

print("\nSelected epoch:", S1_SELECTED_EPOCH)
print("Selected checkpoint:", S1_SELECTED_CHECKPOINT)
print(f"Selected label-tuned Macro-F1: {S1_SELECTED_EVALUATION['label_tuned_macro_f1']:.4f}")
print("Selected per-label thresholds:")
for label in S1_LABELS:
    print(f"  {label:12s} {S1_SELECTED_LABEL_THRESHOLDS[label]:.2f}")

print("\nValidation report:", S1_EVALUATION_SUMMARY_PATH)
print("Note: threshold calibration on 14 validation documents can overfit.")
print("Cell 10 finished. Fold 3 remains untouched.")

S1 REDO — EXACT-SPAN VALIDATION
Validation documents: 14
Exact gold spans: 1861
Labels: 12

Evaluating epoch_1.pt...
  Validation inference: 100/293
  Validation inference: 200/293
  Validation inference: 293/293
  Global threshold=0.90 | Macro-F1=0.0000 | Micro-F1=0.0000
  Label-tuned Macro-F1=0.0000 | Micro-F1=0.0000

Evaluating epoch_2.pt...
  Validation inference: 100/293
  Validation inference: 200/293
  Validation inference: 293/293
  Global threshold=0.15 | Macro-F1=0.0010 | Micro-F1=0.0058
  Label-tuned Macro-F1=0.0010 | Micro-F1=0.0058

Evaluating epoch_3.pt...
  Validation inference: 100/293
  Validation inference: 200/293
  Validation inference: 293/293
  Global threshold=0.45 | Macro-F1=0.1469 | Micro-F1=0.2113
  Label-tuned Macro-F1=0.2245 | Micro-F1=0.2145

S1 CHECKPOINT COMPARISON
Epoch    Global Macro-F1    Tuned Macro-F1     Tuned Micro-F1    
1        0.0000             0.0000             0.0000            
2        0.0010             0.0010             0.0058        

In [11]:
# ============================================================
# S1 REDO — PATCH 1
# Purpose:
#   1. Safely handle empty or whitespace-only test documents.
#   2. Restore Fold 3 texts if S1_TEST_TEXTS is not defined.
#   3. Verify the empty-document case without requiring a fixed ID.
# ============================================================

import json
from pathlib import Path
import numpy as np

# ---------- Restore test texts if absent from the current kernel ----------
if "S1_TEST_TEXTS" not in globals():
    if "S1_TEST_JSON_PATH" in globals():
        s1_test_json_path = Path(S1_TEST_JSON_PATH)
    else:
        s1_test_json_path = Path(
            "/kaggle/working/S1_phase2_hf/Phase_2_LNER/Test/fold3-text.json"
        )

    if not s1_test_json_path.is_file():
        raise FileNotFoundError(
            f"Fold 3 text JSON not found: {s1_test_json_path}. "
            "Run the S1 data-download cell first."
        )

    with open(s1_test_json_path, "r", encoding="utf-8") as f:
        S1_TEST_TEXTS = json.load(f)

    S1_TEST_TEXTS = {
        str(doc_id): text
        for doc_id, text in S1_TEST_TEXTS.items()
    }

# ---------- Empty-document-safe inference window builder ----------
def s1_build_inference_windows(doc_id, text):
    encoding = S1_TOKENIZER.encode(text)
    token_ids = encoding.ids
    offsets = encoding.offsets

    # Empty documents are valid; generate no windows for them.
    if len(token_ids) == 0 and len(offsets) == 0:
        return []

    if len(token_ids) != len(offsets):
        raise ValueError(
            f"{doc_id}: inconsistent tokenizer output "
            f"(tokens={len(token_ids)}, offsets={len(offsets)})."
        )

    if not token_ids:
        raise ValueError(
            f"{doc_id}: tokenizer returned offsets without tokens."
        )

    windows = []
    starts = s1_make_window_starts(
        len(token_ids), S1_WINDOW_TOKENS, S1_WINDOW_STRIDE
    )

    for token_start in starts:
        token_end = min(token_start + S1_WINDOW_TOKENS, len(token_ids))
        local_ids = token_ids[token_start:token_end]
        local_offsets = offsets[token_start:token_end]

        char_start = (
            0 if token_start == 0 else int(local_offsets[0][0])
        )
        char_end = (
            len(text) if token_end == len(token_ids)
            else int(local_offsets[-1][1])
        )
        char_start = max(0, char_start)
        char_end = min(len(text), char_end)

        if char_end <= char_start:
            raise ValueError(
                f"{doc_id}: invalid character range "
                f"({char_start}, {char_end})."
            )

        char_text = text[char_start:char_end]
        char_to_token = s1_char_to_local_token(
            local_offsets, char_start, char_end
        )

        char_ids = np.fromiter(
            (S1_CHAR_TO_ID.get(ch, S1_CHAR_UNK_ID) for ch in char_text),
            dtype=np.int64,
            count=len(char_text),
        )

        windows.append({
            "doc_id": str(doc_id),
            "token_start": token_start,
            "char_start": char_start,
            "char_end": char_end,
            "input_ids": np.asarray(local_ids, dtype=np.int64),
            "char_ids": char_ids,
            "char_to_token": char_to_token.astype(np.int64, copy=False),
            "start_targets": np.zeros(
                (len(char_text), S1_NUM_LABELS), dtype=np.float32
            ),
            "end_targets": np.zeros(
                (len(char_text), S1_NUM_LABELS), dtype=np.float32
            ),
        })

    return windows


print("Empty-document-safe inference function installed.")
print("Fold 3 documents loaded:", len(S1_TEST_TEXTS))

# Test the known empty document when present; otherwise test any empty document.
S1_EMPTY_TEST_IDS = [
    doc_id for doc_id, text in S1_TEST_TEXTS.items()
    if len(text) == 0
]

if "1305957" in S1_TEST_TEXTS:
    S1_CHECK_DOC_ID = "1305957"
elif S1_EMPTY_TEST_IDS:
    S1_CHECK_DOC_ID = S1_EMPTY_TEST_IDS[0]
else:
    S1_CHECK_DOC_ID = None

if S1_CHECK_DOC_ID is not None:
    print("Empty document ID:", S1_CHECK_DOC_ID)
    print("Character length:", len(S1_TEST_TEXTS[S1_CHECK_DOC_ID]))
    print(
        "Inference windows:",
        len(s1_build_inference_windows(
            S1_CHECK_DOC_ID, S1_TEST_TEXTS[S1_CHECK_DOC_ID]
        ))
    )
else:
    print("No completely empty document found in Fold 3.")

Empty-document-safe inference function installed.
Fold 3 documents loaded: 35
Empty document ID: 1305957
Character length: 0
Inference windows: 0


In [12]:
# ============================================================
# S1 REDO — CELL 11
# Purpose:
#   1. Load the epoch-3 checkpoint selected by validation Macro-F1.
#   2. Tokenize Fold 3 without truncation and create overlapping windows.
#   3. Generate character-level entity boundary predictions.
#   4. Decode with the selected validation-calibrated thresholds.
#   5. Validate all document IDs, labels, offsets, and ZIP structure.
# ============================================================

import json
import zipfile
import gc
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

# ---------- Restore the selected checkpoint ----------
S1_FINAL_CHECKPOINT_PATH = Path(S1_SELECTED_CHECKPOINT)
if not S1_FINAL_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Selected checkpoint is missing: {S1_FINAL_CHECKPOINT_PATH}"
    )

S1_FINAL_CHECKPOINT = torch.load(
    S1_FINAL_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

if S1_FINAL_CHECKPOINT.get("labels") != S1_LABEL_TO_ID:
    raise ValueError("Selected checkpoint label mapping does not match.")

S1_LOAD_STATUS = S1_NER_MODEL.load_state_dict(
    S1_FINAL_CHECKPOINT["model_state_dict"],
    strict=True,
)
S1_NER_MODEL.eval()

print("=" * 68)
print("S1 REDO — FOLD 3 INFERENCE")
print("=" * 68)
print("Selected checkpoint:", S1_FINAL_CHECKPOINT_PATH)
print("Selected epoch:", S1_SELECTED_EPOCH)
print(
    "Validation Macro-F1:",
    f"{S1_SELECTED_EVALUATION['label_tuned_macro_f1']:.4f}"
)
print("Strict checkpoint loading:", S1_LOAD_STATUS)


# ---------- Load test text only; do not load test gold labels ----------
with open(S1_TEST_JSON_PATH, "r", encoding="utf-8") as f:
    S1_TEST_TEXTS = json.load(f)

if not isinstance(S1_TEST_TEXTS, dict):
    raise TypeError("Fold 3 input must be a JSON object mapping IDs to text.")

S1_TEST_TEXTS = {str(k): v for k, v in S1_TEST_TEXTS.items()}

if not S1_TEST_TEXTS or not all(
    isinstance(text, str) for text in S1_TEST_TEXTS.values()
):
    raise ValueError("Fold 3 JSON contains missing or invalid document text.")

print("Fold 3 documents:", len(S1_TEST_TEXTS))
print("Fold 3 gold labels accessed: NO")


# ---------- Build inference windows with the same windowing as training ----------
def s1_build_inference_windows(doc_id, text):
    encoding = S1_TOKENIZER.encode(text)
    token_ids = encoding.ids
    offsets = encoding.offsets

    # Empty documents are valid and should produce no entity spans.
    if len(token_ids) == 0 and len(offsets) == 0:
        return []

    if len(token_ids) != len(offsets):
        raise ValueError(
            f"{doc_id}: inconsistent tokenizer output "
            f"(tokens={len(token_ids)}, offsets={len(offsets)})."
        )

    windows = []
    starts = s1_make_window_starts(
        len(token_ids), S1_WINDOW_TOKENS, S1_WINDOW_STRIDE
    )

    for token_start in starts:
        token_end = min(token_start + S1_WINDOW_TOKENS, len(token_ids))
        local_ids = token_ids[token_start:token_end]
        local_offsets = offsets[token_start:token_end]

        char_start = (
            0 if token_start == 0 else int(local_offsets[0][0])
        )
        char_end = (
            len(text) if token_end == len(token_ids)
            else int(local_offsets[-1][1])
        )
        char_start = max(0, char_start)
        char_end = min(len(text), char_end)

        if char_end <= char_start:
            raise ValueError(
                f"{doc_id}: invalid inference character range "
                f"({char_start}, {char_end})."
            )

        char_text = text[char_start:char_end]
        char_to_token = s1_char_to_local_token(
            local_offsets, char_start, char_end
        )

        char_ids = np.fromiter(
            (
                S1_CHAR_TO_ID.get(char, S1_CHAR_UNK_ID)
                for char in char_text
            ),
            dtype=np.int64,
            count=len(char_text),
        )

        windows.append({
            "doc_id": str(doc_id),
            "token_start": token_start,
            "char_start": char_start,
            "char_end": char_end,
            "input_ids": np.asarray(local_ids, dtype=np.int64),
            "char_ids": char_ids,
            "char_to_token": char_to_token.astype(np.int64, copy=False),
            "start_targets": np.zeros(
                (len(char_text), S1_NUM_LABELS), dtype=np.float32
            ),
            "end_targets": np.zeros(
                (len(char_text), S1_NUM_LABELS), dtype=np.float32
            ),
        })

    return windows


S1_TEST_WINDOWS = []
S1_TEST_WINDOW_COUNTS = {}

for doc_id, text in S1_TEST_TEXTS.items():
    doc_windows = s1_build_inference_windows(doc_id, text)
    S1_TEST_WINDOWS.extend(doc_windows)
    S1_TEST_WINDOW_COUNTS[doc_id] = len(doc_windows)

if not S1_TEST_WINDOWS:
    raise RuntimeError("No Fold 3 inference windows were created.")

S1_TEST_LOADER = DataLoader(
    S1NERDataset(S1_TEST_WINDOWS),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=s1_collate_windows,
    pin_memory=(S1_DEVICE.type == "cuda"),
)

print("Inference windows:", len(S1_TEST_WINDOWS))


# ---------- Aggregate predictions over overlapping character windows ----------
S1_TEST_AGGREGATE = {
    doc_id: {
        "start_sum": np.zeros(
            (len(text), S1_NUM_LABELS), dtype=np.float32
        ),
        "end_sum": np.zeros(
            (len(text), S1_NUM_LABELS), dtype=np.float32
        ),
        "count": np.zeros(len(text), dtype=np.float32),
    }
    for doc_id, text in S1_TEST_TEXTS.items()
}

with torch.no_grad():
    for batch_idx, batch in enumerate(S1_TEST_LOADER, start=1):
        batch_device = s1_to_device(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S1_DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = S1_NER_MODEL(
                batch_device["input_ids"],
                batch_device["char_ids"],
                batch_device["char_to_token"],
            )

        start_logits = start_logits.float().cpu().numpy()
        end_logits = end_logits.float().cpu().numpy()

        for i, raw_doc_id in enumerate(batch["doc_ids"]):
            doc_id = str(raw_doc_id)
            char_start = int(batch["char_starts"][i])
            char_end = int(batch["char_ends"][i])
            text_length = len(S1_TEST_TEXTS[doc_id])

            n = min(
                char_end - char_start,
                start_logits.shape[1],
                end_logits.shape[1],
                max(0, text_length - char_start),
            )
            if n <= 0:
                continue

            d = S1_TEST_AGGREGATE[doc_id]
            d["start_sum"][char_start:char_start+n] += start_logits[i, :n]
            d["end_sum"][char_start:char_start+n] += end_logits[i, :n]
            d["count"][char_start:char_start+n] += 1.0

        if batch_idx % 100 == 0 or batch_idx == len(S1_TEST_LOADER):
            print(f"Inference: {batch_idx}/{len(S1_TEST_LOADER)} windows")

# Average overlapping logits, then apply sigmoid.
for doc_id, d in S1_TEST_AGGREGATE.items():
    denominator = np.maximum(d["count"][:, None], 1.0)
    avg_start = d["start_sum"] / denominator
    avg_end = d["end_sum"] / denominator

    d["start"] = 1.0 / (
        1.0 + np.exp(-np.clip(avg_start, -30, 30))
    )
    d["end"] = 1.0 / (
        1.0 + np.exp(-np.clip(avg_end, -30, 30))
    )

    del d["start_sum"], d["end_sum"]


# ---------- Decode using validation-selected label thresholds ----------
S1_PREDICTIONS = {doc_id: [] for doc_id in S1_TEST_TEXTS}

for doc_id, d in S1_TEST_AGGREGATE.items():
    text_length = len(S1_TEST_TEXTS[doc_id])
    document_spans = set()

    for label in S1_LABELS:
        label_id = S1_LABEL_TO_ID[label]
        threshold = float(S1_SELECTED_LABEL_THRESHOLDS[label])

        starts = np.flatnonzero(d["start"][:, label_id] >= threshold)
        ends = np.flatnonzero(d["end"][:, label_id] >= threshold)

        # Use the same greedy boundary pairing strategy as validation.
        end_idx = 0
        for start in starts:
            while end_idx < len(ends) and ends[end_idx] < start:
                end_idx += 1

            if end_idx >= len(ends):
                break

            # The predicted end boundary is inclusive; submission end is exclusive.
            end_exclusive = int(ends[end_idx]) + 1
            end_idx += 1

            if 0 <= int(start) < end_exclusive <= text_length:
                document_spans.add((
                    int(start), end_exclusive, label
                ))

    S1_PREDICTIONS[doc_id] = [
        [start, end, label]
        for start, end, label in sorted(
            document_spans, key=lambda span: (span[0], span[1], span[2])
        )
    ]


# ---------- Validate schema, offsets, IDs, and duplicates ----------
S1_OUTPUT_DIR = Path("/kaggle/working/S1_A_FullFineTuning")
S1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S1_PREDICTION_JSON_PATH = S1_OUTPUT_DIR / "prediction.json"
S1_SUBMISSION_ZIP_PATH = Path(
    "/kaggle/working/S1_A_FullFineTuning_Submission.zip"
)

S1_VALIDATION_ERRORS = []
S1_TOTAL_PREDICTED_SPANS = 0

for doc_id, spans in S1_PREDICTIONS.items():
    text_length = len(S1_TEST_TEXTS[doc_id])
    seen = set()

    for span in spans:
        valid_structure = (
            isinstance(span, list)
            and len(span) == 3
            and isinstance(span[0], int)
            and isinstance(span[1], int)
            and isinstance(span[2], str)
        )
        if not valid_structure:
            S1_VALIDATION_ERRORS.append((doc_id, span, "invalid structure"))
            continue

        start, end, label = span
        if not (0 <= start < end <= text_length):
            S1_VALIDATION_ERRORS.append((doc_id, span, "invalid offsets"))
        if label not in S1_LABEL_TO_ID:
            S1_VALIDATION_ERRORS.append((doc_id, span, "invalid label"))
        if tuple(span) in seen:
            S1_VALIDATION_ERRORS.append((doc_id, span, "duplicate span"))
        seen.add(tuple(span))

        S1_TOTAL_PREDICTED_SPANS += 1

if S1_VALIDATION_ERRORS:
    raise ValueError(
        f"Invalid predictions found: {S1_VALIDATION_ERRORS[:5]}"
    )

# Write the official format: document ID -> [start, end, label] triples.
with open(S1_PREDICTION_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(S1_PREDICTIONS, f, ensure_ascii=False)

# Put prediction.json at the ZIP root.
with zipfile.ZipFile(
    S1_SUBMISSION_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(S1_PREDICTION_JSON_PATH, arcname="prediction.json")

# Independently verify the packaged JSON and exact ID coverage.
with zipfile.ZipFile(S1_SUBMISSION_ZIP_PATH, "r") as zf:
    if zf.namelist() != ["prediction.json"]:
        raise AssertionError(f"Unexpected ZIP contents: {zf.namelist()}")

    with zf.open("prediction.json") as f:
        S1_PACKAGED_PREDICTIONS = json.load(f)

if set(S1_PACKAGED_PREDICTIONS) != set(S1_TEST_TEXTS):
    raise AssertionError("Submission document IDs do not match Fold 3.")

if any(not isinstance(v, list) for v in S1_PACKAGED_PREDICTIONS.values()):
    raise AssertionError("One or more document predictions are not lists.")

print("\n" + "=" * 68)
print("S1 REDO — SUBMISSION VERIFIED")
print("=" * 68)
print("Selected epoch:", S1_SELECTED_EPOCH)
print(
    "Validation label-tuned Macro-F1:",
    f"{S1_SELECTED_EVALUATION['label_tuned_macro_f1']:.4f}"
)
print("Fold 3 documents:", len(S1_PREDICTIONS))
print("Total predicted spans:", S1_TOTAL_PREDICTED_SPANS)
print(
    "Documents with predictions:",
    sum(bool(spans) for spans in S1_PREDICTIONS.values())
)
print(
    "Documents with no predictions:",
    sum(not spans for spans in S1_PREDICTIONS.values())
)
print("Invalid spans:", len(S1_VALIDATION_ERRORS))
print("Fold 3 gold labels accessed: NO")
print("Prediction JSON:", S1_PREDICTION_JSON_PATH)
print("Submission ZIP:", S1_SUBMISSION_ZIP_PATH)
print("ZIP root contents: prediction.json")

# Release large inference intermediates while keeping the final predictions.
del S1_TEST_LOADER, S1_TEST_WINDOWS, S1_TEST_AGGREGATE
del S1_FINAL_CHECKPOINT
gc.collect()
if S1_DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\nCell 11 finished.")

S1 REDO — FOLD 3 INFERENCE
Selected checkpoint: /kaggle/working/S1_A_FullFineTuning/checkpoints/epoch_3.pt
Selected epoch: 3
Validation Macro-F1: 0.2245
Strict checkpoint loading: <All keys matched successfully>
Fold 3 documents: 35
Fold 3 gold labels accessed: NO
Inference windows: 637
Inference: 100/637 windows
Inference: 200/637 windows
Inference: 300/637 windows
Inference: 400/637 windows
Inference: 500/637 windows
Inference: 600/637 windows
Inference: 637/637 windows

S1 REDO — SUBMISSION VERIFIED
Selected epoch: 3
Validation label-tuned Macro-F1: 0.2245
Fold 3 documents: 35
Total predicted spans: 7120
Documents with predictions: 34
Documents with no predictions: 1
Invalid spans: 0
Fold 3 gold labels accessed: NO
Prediction JSON: /kaggle/working/S1_A_FullFineTuning/prediction.json
Submission ZIP: /kaggle/working/S1_A_FullFineTuning_Submission.zip
ZIP root contents: prediction.json

Cell 11 finished.
